# 13_graph_ablation_and_spatial_sensitivity_analysis.ipynb

## Σκοπός του notebook

Το παρόν notebook αποτελεί **strict follow-up** του `NB12`
και όχι νέο broad graph-benchmark stage.

Στόχος του είναι να εξετάσει, με **benchmark-safe** και **non-overclaiming**
τρόπο, αν η spatial graph πληροφορία προσθέτει ουσιαστική forecasting αξία
στο current repository setup και πόσο ευαίσθητο είναι το αποτέλεσμα
σε controlled graph / aggregation choices.

Πιο συγκεκριμένα, το `NB13` στοχεύει να ελέγξει:

- αν η spatial graph πληροφορία βοηθά ουσιαστικά σε σχέση με ένα no-spatial control,
- αν το αποτέλεσμα είναι ευαίσθητο στην επιλογή graph topology variant,
- αν η aggregation depth επηρεάζει αρνητικά ή θετικά την επίδοση,
- αν υπάρχουν ενδείξεις oversmoothing ή neighbor contamination,
- ή αν το forecasting πρόβλημα κυριαρχείται ήδη από πολύ ισχυρό tabular signal.

## Τι ΔΕΝ κάνει το notebook

Το `NB13`:

- **δεν** ανοίγει νέο broad GNN benchmark scope,
- **δεν** εισάγει sequence stage,
- **δεν** εισάγει Mamba / Graph-Mamba,
- **δεν** μεταβάλλει το canonical `baseline_metrics.csv`,
- **δεν** χρησιμοποιεί test split για model selection,
- **δεν** κάνει claim graph superiority,
- **δεν** μετατρέπει το forecasting notebook σε PHM / fault analysis stage.

## Μεθοδολογική θέση

Το `NB12` απέδειξε ότι το repository διαθέτει πλέον λειτουργικό,
reproducible και benchmark-safe graph-based forecasting baseline,
αλλά **δεν** απέδειξε υπεροχή του graph approach έναντι του canonical tabular backbone.

Άρα το σωστό επόμενο βήμα είναι ένα **controlled ablation / spatial sensitivity notebook**
που θα παραμείνει:

- forecasting-first,
- benchmark-safe,
- validation-driven,
- read-only ως προς τα canonical benchmark artifacts,
- και αυστηρό ως προς scientific interpretation.

## Evaluation discipline

Στο παρόν notebook:

- το **validation split** χρησιμοποιείται μόνο για model selection,
- το **test split** χρησιμοποιείται μόνο για final reporting,
- όλα τα benchmark references διαβάζονται ως **read-only**,
- και κάθε variant δηλώνεται εκ των προτέρων σε explicit experiment registry.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
from copy import deepcopy
import importlib
import json
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display

# ============================================================
# Fail-fast import policy για PyTorch Geometric
# ============================================================
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GCNConv
except Exception as exc:
    raise ImportError(
        "Το NB13 απαιτεί εγκατεστημένο το torch_geometric. "
        "Επιβεβαίωσε ότι το environment του repository έχει το σωστό dependency stack."
    ) from exc


# ============================================================
# Reproducibility policy
# ============================================================
SEED = 42


def set_global_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_global_reproducibility(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# Project-root / path helpers
# ============================================================
def find_project_root(start_path: Path) -> Path:
    """
    Εντοπίζει το repository root ανεβαίνοντας από το current working directory.
    """
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "Δεν βρέθηκε project root με φακέλους `data/` και `notebooks/`."
    )


def resolve_under_root(path_like: str | Path, root: Path) -> Path:
    """
    Αν το path είναι σχετικό, το επιλύουμε κάτω από το project root.
    """
    path_obj = Path(path_like)
    return path_obj if path_obj.is_absolute() else (root / path_obj)


ROOT = find_project_root(Path.cwd())

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

try:
    cfg = importlib.import_module("src.config")
except Exception:
    cfg = None


def cfg_get(name: str, default: Any) -> Any:
    """
    Διαβάζει attribute από το src.config μόνο αν υπάρχει.
    """
    if cfg is None:
        return default
    return getattr(cfg, name, default)


# ============================================================
# Canonical repository constants
# ============================================================
PARK_ID_COLUMN = cfg_get("PARK_ID_COLUMN", "park_id")
TIMESTAMP_COLUMN = cfg_get("TIMESTAMP_COLUMN", "timestamp")
TARGET_COLUMN = cfg_get("TARGET_COLUMN", "Power_Output_Normalized")
BASELINE_COLUMN = cfg_get("BASELINE_COLUMN", "Baseline_Prediction")


# ============================================================
# Canonical upstream artifacts (NB11)
# ============================================================
NB11_EXPORT_DIR = resolve_under_root(
    cfg_get("NB11_EXPORT_DIR", "data/processed/graph_packaging"),
    ROOT,
)

TRAIN_GRAPH_PACKAGE_PATH = resolve_under_root(
    cfg_get("NB11_TRAIN_GRAPH_DATASET", NB11_EXPORT_DIR / "train_graph_dataset.pt"),
    ROOT,
)
VAL_GRAPH_PACKAGE_PATH = resolve_under_root(
    cfg_get("NB11_VAL_GRAPH_DATASET", NB11_EXPORT_DIR / "val_graph_dataset.pt"),
    ROOT,
)
TEST_GRAPH_PACKAGE_PATH = resolve_under_root(
    cfg_get("NB11_TEST_GRAPH_DATASET", NB11_EXPORT_DIR / "test_graph_dataset.pt"),
    ROOT,
)

NB11_FEATURE_ROLE_MANIFEST_PATH = resolve_under_root(
    cfg_get("NB11_FEATURE_ROLE_MANIFEST", NB11_EXPORT_DIR / "nb11_feature_role_manifest.csv"),
    ROOT,
)
NB11_NODE_FEATURE_MANIFEST_PATH = resolve_under_root(
    cfg_get("NB11_NODE_FEATURE_MANIFEST", NB11_EXPORT_DIR / "nb11_node_feature_manifest.csv"),
    ROOT,
)


# ============================================================
# Read-only benchmark references
# ============================================================
BASELINE_METRICS_PATH = resolve_under_root(
    cfg_get("BASELINE_METRICS_PATH", "data/processed/baseline_metrics.csv"),
    ROOT,
)

NB12_REFERENCE_DIR = resolve_under_root(
    "data/processed/graph_baselines/nb12_first_graph_baseline",
    ROOT,
)

NB12_TEST_METRICS_PATH = NB12_REFERENCE_DIR / "nb12_test_metrics.csv"
NB12_BENCHMARK_COMPARISON_PATH = NB12_REFERENCE_DIR / "nb12_benchmark_comparison.csv"
NB12_RUN_CONFIG_PATH = NB12_REFERENCE_DIR / "nb12_run_config.json"
NB12_TRAINING_HISTORY_PATH = NB12_REFERENCE_DIR / "nb12_training_history.csv"


# ============================================================
# NB13 export namespace
# ============================================================
NB13_EXPORT_DIR = resolve_under_root(
    "data/processed/graph_baselines/nb13_graph_ablation_and_spatial_sensitivity",
    ROOT,
)
NB13_EXPORT_DIR.mkdir(parents=True, exist_ok=True)


print("SEED:", SEED)
print("DEVICE:", DEVICE)
print("ROOT:", ROOT)
print("NB11_EXPORT_DIR:", NB11_EXPORT_DIR)
print("NB12_REFERENCE_DIR:", NB12_REFERENCE_DIR)
print("NB13_EXPORT_DIR:", NB13_EXPORT_DIR)

SEED: 42
DEVICE: cpu
ROOT: C:\Users\diony\Desktop\WindPower_DigitalTwin
NB11_EXPORT_DIR: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_packaging
NB12_REFERENCE_DIR: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb12_first_graph_baseline
NB13_EXPORT_DIR: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity


## 2. Canonical artifacts, read-only NB12 references και fail-fast integrity checks

Το `NB13` βασίζεται αποκλειστικά σε canonical upstream artifacts και δεν επανανοίγει
raw-validation, feature-engineering ή split-definition scope.

Συγκεκριμένα:

- το `NB11` παρέχει το canonical graph-model input packaging layer,
- το `NB12` παρέχει το πρώτο actual graph-based forecasting reference run,
- και το `NB13` λειτουργεί ως strict follow-up ablation / spatial sensitivity notebook.

### Μεθοδολογική απαίτηση

Πριν οριστεί οποιοδήποτε ablation experiment, πρέπει να επιβεβαιωθεί ότι:

- τα required `NB11` packaged artifacts υπάρχουν και φορτώνονται σωστά,
- τα `NB12` reference artifacts υπάρχουν και διαβάζονται μόνο ως read-only,
- η graph snapshot structure είναι συνεπής across train / validation / test,
- και το current notebook δεν βασίζεται σε implicit assumptions για shapes, fields ή row counts.

### Read-only benchmark discipline

Στο παρόν notebook:

- το `data/processed/baseline_metrics.csv` αντιμετωπίζεται ως canonical benchmark authority,
- τα `NB12` exports χρησιμοποιούνται μόνο ως reference / comparison layer,
- και κανένα upstream benchmark artifact δεν μεταβάλλεται από το `NB13`.

In [3]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from torch_geometric.data import Data

from src import config as cfg


# ============================================================
# Strict artifact loading helpers
# ============================================================
def assert_path_exists(path, label: str) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"[MISSING] {label}: {path}")


def read_csv_strict(path, label: str) -> pd.DataFrame:
    path = Path(path)
    assert_path_exists(path, label)

    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"[EMPTY CSV] {label}: {path}")
    if df.columns.duplicated().any():
        duplicated = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"[DUPLICATE COLUMNS] {label}: {duplicated}")

    return df


def read_json_strict(path, label: str) -> dict[str, Any]:
    path = Path(path)
    assert_path_exists(path, label)

    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if not isinstance(payload, dict):
        raise TypeError(
            f"[INVALID JSON TYPE] {label}: expected dict, got {type(payload)}"
        )

    return payload


def torch_load_strict(path, label: str):
    path = Path(path)
    assert_path_exists(path, label)

    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(path, map_location="cpu")

    return obj


# ============================================================
# NB11 packaged graph dataset schema
# ============================================================
REQUIRED_PACKAGE_KEYS = {
    "split_name",
    "node_ids",
    "timestamps",
    "edge_index",
    "edge_attr_km",
    "static_x",
    "dynamic_x",
    "target_y",
    "baseline_reference",
    "observed_mask",
    "static_feature_names",
    "dynamic_feature_names",
    "target_name",
    "baseline_name",
    "n_nodes",
    "n_timestamps",
    "n_static_features",
    "n_dynamic_features",
}


def validate_package_type(package: Any, label: str) -> None:
    if not isinstance(package, dict):
        raise TypeError(
            f"[INVALID PACKAGE TYPE] {label}: expected dict, got {type(package)}"
        )


def assert_required_package_keys(package: dict[str, Any], label: str) -> None:
    missing = sorted(REQUIRED_PACKAGE_KEYS - set(package.keys()))
    if missing:
        raise KeyError(f"[MISSING PACKAGE KEYS] {label}: {missing}")


def assert_package_split_name(
    package: dict[str, Any],
    expected_split_name: str,
    label: str,
) -> None:
    actual_split_name = package.get("split_name", None)
    if actual_split_name != expected_split_name:
        raise ValueError(
            f"[SPLIT NAME MISMATCH] {label}: "
            f"expected={expected_split_name}, found={actual_split_name}"
        )


def assert_package_tensor_shapes(package: dict[str, Any], label: str) -> None:
    static_x = package["static_x"]
    dynamic_x = package["dynamic_x"]
    target_y = package["target_y"]
    baseline_reference = package["baseline_reference"]
    observed_mask = package["observed_mask"]
    edge_index = package["edge_index"]
    edge_attr_km = package["edge_attr_km"]

    if static_x.ndim != 2:
        raise ValueError(f"[INVALID SHAPE] {label}: static_x must be 2D.")
    if dynamic_x.ndim != 3:
        raise ValueError(f"[INVALID SHAPE] {label}: dynamic_x must be 3D [T, N, D].")
    if target_y.ndim != 2:
        raise ValueError(f"[INVALID SHAPE] {label}: target_y must be 2D [T, N].")
    if baseline_reference.ndim != 2:
        raise ValueError(
            f"[INVALID SHAPE] {label}: baseline_reference must be 2D [T, N]."
        )
    if observed_mask.ndim != 2:
        raise ValueError(f"[INVALID SHAPE] {label}: observed_mask must be 2D [T, N].")
    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError(f"[INVALID SHAPE] {label}: edge_index must be shape [2, E].")
    if edge_attr_km.ndim != 2:
        raise ValueError(f"[INVALID SHAPE] {label}: edge_attr_km must be 2D [E, A].")

    if tuple(observed_mask.shape) != tuple(target_y.shape):
        raise ValueError(
            f"[SHAPE MISMATCH] {label}: observed_mask and target_y differ."
        )
    if tuple(baseline_reference.shape) != tuple(target_y.shape):
        raise ValueError(
            f"[SHAPE MISMATCH] {label}: baseline_reference and target_y differ."
        )
    if dynamic_x.shape[0] != target_y.shape[0]:
        raise ValueError(
            f"[SHAPE MISMATCH] {label}: dynamic_x and target_y differ on T."
        )
    if dynamic_x.shape[1] != target_y.shape[1]:
        raise ValueError(
            f"[SHAPE MISMATCH] {label}: dynamic_x and target_y differ on N."
        )
    if static_x.shape[0] != target_y.shape[1]:
        raise ValueError(f"[SHAPE MISMATCH] {label}: static_x node count mismatch.")
    if edge_attr_km.shape[0] != edge_index.shape[1]:
        raise ValueError(
            f"[SHAPE MISMATCH] {label}: edge_attr_km row count != edge count."
        )

    if int(observed_mask.sum().item()) == 0:
        raise ValueError(f"[EMPTY OBSERVED SPACE] {label}: no observed entries found.")

    observed_per_timestamp = observed_mask.sum(dim=1)
    if torch.any(observed_per_timestamp == 0):
        raise ValueError(
            f"[INVALID COVERAGE] {label}: at least one timestamp has zero observed nodes."
        )


def assert_observed_entries_are_not_nan(package: dict[str, Any], label: str) -> None:
    observed_mask = package["observed_mask"].cpu()
    target_y = package["target_y"].cpu()
    baseline_reference = package["baseline_reference"].cpu()
    dynamic_x = package["dynamic_x"].cpu()

    if torch.isnan(target_y)[observed_mask].any():
        raise ValueError(f"[NaN OBSERVED TARGET] {label}")

    if torch.isnan(baseline_reference)[observed_mask].any():
        raise ValueError(f"[NaN OBSERVED BASELINE] {label}")

    observed_mask_3d = observed_mask.unsqueeze(-1).expand_as(dynamic_x)
    if torch.isnan(dynamic_x)[observed_mask_3d].any():
        raise ValueError(f"[NaN OBSERVED DYNAMIC_X] {label}")


# ============================================================
# TemporalGraphSnapshotDataset
# ============================================================
class TemporalGraphSnapshotDataset(torch.utils.data.Dataset):
    """
    Μετατρέπει ένα NB11 packaged split σε snapshot-level PyG dataset.

    Κάθε item αντιστοιχεί σε ένα timestamp graph snapshot.
    """

    def __init__(
        self,
        package: dict[str, Any],
        include_observed_indicator: bool = True,
        fill_value: float = 0.0,
    ) -> None:
        self.package = package
        self.include_observed_indicator = include_observed_indicator
        self.fill_value = float(fill_value)

        self.timestamps = list(package["timestamps"])
        self.node_ids = list(package["node_ids"])

        self.input_feature_names = (
            list(package["static_feature_names"])
            + list(package["dynamic_feature_names"])
        )

        if self.include_observed_indicator:
            self.input_feature_names = self.input_feature_names + ["observed_indicator"]

        self.n_input_features = len(self.input_feature_names)

    def __len__(self) -> int:
        return int(self.package["n_timestamps"])

    def _build_x(self, t_index: int) -> torch.Tensor:
        static_x = self.package["static_x"]                       # [N, S]
        dynamic_x_t = self.package["dynamic_x"][t_index]         # [N, D]
        observed_mask_t = self.package["observed_mask"][t_index] # [N]

        dynamic_x_t = torch.nan_to_num(
            dynamic_x_t,
            nan=self.fill_value,
            posinf=self.fill_value,
            neginf=self.fill_value,
        )

        parts = [static_x, dynamic_x_t]

        if self.include_observed_indicator:
            observed_indicator = observed_mask_t.to(torch.float32).unsqueeze(-1)
            parts.append(observed_indicator)

        x = torch.cat(parts, dim=1)

        if x.ndim != 2:
            raise ValueError(f"Expected 2D x matrix, got shape={tuple(x.shape)}")

        return x

    def __getitem__(self, index: int) -> Data:
        if not (0 <= index < len(self)):
            raise IndexError(f"Index out of range: {index}")

        x = self._build_x(index)

        data = Data(
            x=x,
            edge_index=self.package["edge_index"],
            y=self.package["target_y"][index],
        )

        data.edge_attr = self.package["edge_attr_km"]
        data.observed_mask = self.package["observed_mask"][index]
        data.baseline_reference = self.package["baseline_reference"][index]
        data.time_index = torch.tensor([index], dtype=torch.long)
        data.num_nodes = int(self.package["n_nodes"])

        return data


def summarize_package(package: dict[str, Any], split_name: str) -> dict[str, Any]:
    observed_per_timestamp = package["observed_mask"].sum(dim=1).cpu().numpy()

    return {
        "split": split_name,
        "package_type": type(package).__name__,
        "n_timestamps": int(package["n_timestamps"]),
        "n_nodes": int(package["n_nodes"]),
        "n_static_features": int(package["n_static_features"]),
        "n_dynamic_features": int(package["n_dynamic_features"]),
        "partial_timestamp_count": int(package.get("partial_timestamp_count", -1)),
        "full_coverage_timestamp_count": int(package.get("full_coverage_timestamp_count", -1)),
        "min_observed_nodes_per_timestamp": int(observed_per_timestamp.min()),
        "max_observed_nodes_per_timestamp": int(observed_per_timestamp.max()),
        "mean_observed_nodes_per_timestamp": float(observed_per_timestamp.mean()),
    }


def summarize_dataset_first_snapshot(
    dataset: TemporalGraphSnapshotDataset,
    split_name: str,
) -> dict[str, Any]:
    sample = dataset[0]

    return {
        "split": split_name,
        "dataset_length": len(dataset),
        "n_input_features": int(dataset.n_input_features),
        "first_x_shape": str(tuple(sample.x.shape)),
        "first_y_shape": str(tuple(sample.y.shape)),
        "first_edge_index_shape": str(tuple(sample.edge_index.shape)),
        "first_edge_attr_shape": str(tuple(sample.edge_attr.shape)),
        "first_observed_mask_shape": str(tuple(sample.observed_mask.shape)),
    }


# ============================================================
# 1. Required artifact paths
# ============================================================
required_paths = {
    "BASELINE_METRICS_PATH": cfg.BASELINE_METRICS_PATH,
    "NB11_FEATURE_ROLE_MANIFEST": cfg.NB11_FEATURE_ROLE_MANIFEST,
    "NB11_NODE_FEATURE_MANIFEST": cfg.NB11_NODE_FEATURE_MANIFEST,
    "NB11_SPLIT_GRAPH_PACKAGING_SUMMARY": cfg.NB11_SPLIT_GRAPH_PACKAGING_SUMMARY,
    "NB11_PACKAGING_STATUS_MANIFEST": cfg.NB11_PACKAGING_STATUS_MANIFEST,
    "NB11_TRAIN_TIMESTAMP_COVERAGE": cfg.NB11_TRAIN_TIMESTAMP_COVERAGE,
    "NB11_VAL_TIMESTAMP_COVERAGE": cfg.NB11_VAL_TIMESTAMP_COVERAGE,
    "NB11_TEST_TIMESTAMP_COVERAGE": cfg.NB11_TEST_TIMESTAMP_COVERAGE,
    "NB11_TRAIN_GRAPH_DATASET": cfg.NB11_TRAIN_GRAPH_DATASET,
    "NB11_VAL_GRAPH_DATASET": cfg.NB11_VAL_GRAPH_DATASET,
    "NB11_TEST_GRAPH_DATASET": cfg.NB11_TEST_GRAPH_DATASET,
    "NB12_BENCHMARK_COMPARISON": cfg.NB12_BENCHMARK_COMPARISON,
    "NB12_RUN_CONFIG": cfg.NB12_RUN_CONFIG,
    "NB12_TEST_METRICS": cfg.NB12_TEST_METRICS,
    "NB12_TRAINING_HISTORY": cfg.NB12_TRAINING_HISTORY,
    "NB12_TEST_PREDICTIONS_OBSERVED_ONLY": cfg.NB12_TEST_PREDICTIONS_OBSERVED_ONLY,
}

for label, path in required_paths.items():
    assert_path_exists(path, label)


# ============================================================
# 2. Load read-only tabular / metadata artifacts
# ============================================================
baseline_metrics_df = read_csv_strict(
    cfg.BASELINE_METRICS_PATH,
    "Canonical baseline metrics",
)

nb11_feature_role_manifest_df = read_csv_strict(
    cfg.NB11_FEATURE_ROLE_MANIFEST,
    "NB11 feature role manifest",
)
nb11_node_feature_manifest_df = read_csv_strict(
    cfg.NB11_NODE_FEATURE_MANIFEST,
    "NB11 node feature manifest",
)
nb11_split_graph_packaging_summary_df = read_csv_strict(
    cfg.NB11_SPLIT_GRAPH_PACKAGING_SUMMARY,
    "NB11 split graph packaging summary",
)
nb11_packaging_status_manifest_df = read_csv_strict(
    cfg.NB11_PACKAGING_STATUS_MANIFEST,
    "NB11 packaging status manifest",
)

nb11_train_timestamp_coverage_df = read_csv_strict(
    cfg.NB11_TRAIN_TIMESTAMP_COVERAGE,
    "NB11 train timestamp coverage",
)
nb11_val_timestamp_coverage_df = read_csv_strict(
    cfg.NB11_VAL_TIMESTAMP_COVERAGE,
    "NB11 val timestamp coverage",
)
nb11_test_timestamp_coverage_df = read_csv_strict(
    cfg.NB11_TEST_TIMESTAMP_COVERAGE,
    "NB11 test timestamp coverage",
)

nb12_benchmark_comparison_df = read_csv_strict(
    cfg.NB12_BENCHMARK_COMPARISON,
    "NB12 benchmark comparison",
)
nb12_test_metrics_df = read_csv_strict(
    cfg.NB12_TEST_METRICS,
    "NB12 test metrics",
)
nb12_training_history_df = read_csv_strict(
    cfg.NB12_TRAINING_HISTORY,
    "NB12 training history",
)
nb12_test_predictions_observed_only_df = read_csv_strict(
    cfg.NB12_TEST_PREDICTIONS_OBSERVED_ONLY,
    "NB12 observed-only test predictions",
)

nb12_run_config = read_json_strict(
    cfg.NB12_RUN_CONFIG,
    "NB12 run config",
)


# ============================================================
# 3. Load NB11 packaged graph artifacts
# ============================================================
train_package = torch_load_strict(
    cfg.NB11_TRAIN_GRAPH_DATASET,
    "NB11 train graph package",
)
val_package = torch_load_strict(
    cfg.NB11_VAL_GRAPH_DATASET,
    "NB11 val graph package",
)
test_package = torch_load_strict(
    cfg.NB11_TEST_GRAPH_DATASET,
    "NB11 test graph package",
)

for split_name, package in {
    "train": train_package,
    "val": val_package,
    "test": test_package,
}.items():
    validate_package_type(package, f"{split_name} package")
    assert_required_package_keys(package, f"{split_name} package")
    assert_package_split_name(package, split_name, f"{split_name} package")
    assert_package_tensor_shapes(package, f"{split_name} package")
    assert_observed_entries_are_not_nan(package, f"{split_name} package")


# ============================================================
# 4. Cross-split package contract checks
# ============================================================
if train_package["node_ids"] != val_package["node_ids"] or train_package["node_ids"] != test_package["node_ids"]:
    raise ValueError("[NODE ORDER MISMATCH] train / val / test node_ids differ.")

if train_package["static_feature_names"] != val_package["static_feature_names"] or train_package["static_feature_names"] != test_package["static_feature_names"]:
    raise ValueError("[STATIC FEATURE MISMATCH] static feature names differ across splits.")

if train_package["dynamic_feature_names"] != val_package["dynamic_feature_names"] or train_package["dynamic_feature_names"] != test_package["dynamic_feature_names"]:
    raise ValueError("[DYNAMIC FEATURE MISMATCH] dynamic feature names differ across splits.")

if train_package["target_name"] != val_package["target_name"] or train_package["target_name"] != test_package["target_name"]:
    raise ValueError("[TARGET NAME MISMATCH] target_name differs across splits.")

if train_package["baseline_name"] != val_package["baseline_name"] or train_package["baseline_name"] != test_package["baseline_name"]:
    raise ValueError("[BASELINE NAME MISMATCH] baseline_name differs across splits.")

expected_n_nodes = int(train_package["n_nodes"])
expected_input_feature_dim = (
    int(train_package["n_static_features"])
    + int(train_package["n_dynamic_features"])
    + 1  # observed_indicator
)

for split_name, package in {
    "val": val_package,
    "test": test_package,
}.items():
    if int(package["n_nodes"]) != expected_n_nodes:
        raise ValueError(
            f"[NODE COUNT MISMATCH] train has {expected_n_nodes}, "
            f"{split_name} has {int(package['n_nodes'])}"
        )


# ============================================================
# 5. Build snapshot-level datasets exactly as NB12 expects
# ============================================================
train_dataset = TemporalGraphSnapshotDataset(train_package)
val_dataset = TemporalGraphSnapshotDataset(val_package)
test_dataset = TemporalGraphSnapshotDataset(test_package)

package_summary_df = pd.DataFrame(
    [
        summarize_package(train_package, "train"),
        summarize_package(val_package, "val"),
        summarize_package(test_package, "test"),
    ]
)

dataset_summary_df = pd.DataFrame(
    [
        summarize_dataset_first_snapshot(train_dataset, "train"),
        summarize_dataset_first_snapshot(val_dataset, "val"),
        summarize_dataset_first_snapshot(test_dataset, "test"),
    ]
)

display(package_summary_df)
display(dataset_summary_df)


# ============================================================
# 6. Reconciliation against NB12 run config
# ============================================================
assert len(train_dataset) == int(nb12_run_config["n_train_snapshots"]), (
    f"Train snapshot mismatch: {len(train_dataset)} vs {nb12_run_config['n_train_snapshots']}"
)
assert len(val_dataset) == int(nb12_run_config["n_val_snapshots"]), (
    f"Validation snapshot mismatch: {len(val_dataset)} vs {nb12_run_config['n_val_snapshots']}"
)
assert len(test_dataset) == int(nb12_run_config["n_test_snapshots"]), (
    f"Test snapshot mismatch: {len(test_dataset)} vs {nb12_run_config['n_test_snapshots']}"
)
assert expected_n_nodes == int(nb12_run_config["n_nodes"]), (
    f"Node count mismatch: {expected_n_nodes} vs {nb12_run_config['n_nodes']}"
)
assert train_dataset.n_input_features == int(nb12_run_config["input_feature_dim"]), (
    f"Feature dimension mismatch: {train_dataset.n_input_features} vs "
    f"{nb12_run_config['input_feature_dim']}"
)


# ============================================================
# 7. Final integrity status
# ============================================================
print("Canonical baseline benchmark table loaded in read-only mode.")
print("\nNB13 PACKAGE INTEGRITY CHECK PASSED")
print("-" * 60)
print(f"Train snapshots      : {len(train_dataset)}")
print(f"Validation snapshots : {len(val_dataset)}")
print(f"Test snapshots       : {len(test_dataset)}")
print(f"Node count           : {expected_n_nodes}")
print(f"Input feature dim    : {train_dataset.n_input_features}")
print(f"NB12 best epoch      : {nb12_run_config.get('best_epoch', '<missing>')}")
print(f"NB12 best val loss   : {nb12_run_config.get('best_val_loss', '<missing>')}")

,split,package_type,n_timestamps,n_nodes,n_static_features,n_dynamic_features,partial_timestamp_count,full_coverage_timestamp_count,min_observed_nodes_per_timestamp,max_observed_nodes_per_timestamp,mean_observed_nodes_per_timestamp
0,train,dict,7819,256,5,36,6494,1325,244,256,253.579230
1,val,dict,720,256,5,36,593,127,249,256,254.163889
2,test,dict,4295,256,5,36,4096,199,245,256,252.930384


,split,dataset_length,n_input_features,first_x_shape,first_y_shape,first_edge_index_shape,first_edge_attr_shape,first_observed_mask_shape
0,train,7819,42,"(256, 42)","(256,)","(2, 1068)","(1068, 1)","(256,)"
1,val,720,42,"(256, 42)","(256,)","(2, 1068)","(1068, 1)","(256,)"
2,test,4295,42,"(256, 42)","(256,)","(2, 1068)","(1068, 1)","(256,)"


Canonical baseline benchmark table loaded in read-only mode.

NB13 PACKAGE INTEGRITY CHECK PASSED
------------------------------------------------------------
Train snapshots      : 7819
Validation snapshots : 720
Test snapshots       : 4295
Node count           : 256
Input feature dim    : 42
NB12 best epoch      : 5
NB12 best val loss   : 0.06048211165600353


## 3. Predeclared experiment registry για controlled graph ablations

Το `NB13` δεν πρέπει να λειτουργήσει ως ad hoc exploration notebook.
Αντίθετα, κάθε run πρέπει να δηλωθεί **εκ των προτέρων** σε explicit experiment registry.

### Γιατί χρειάζεται registry-first λογική

Η graph ablation / spatial sensitivity analysis έχει νόημα μόνο αν:

- οι πειραματικές παραλλαγές είναι δηλωμένες πριν από το training,
- η training configuration παραμένει κατά το δυνατόν σταθερή,
- αλλάζουν μόνο οι παράγοντες που θέλουμε να εξετάσουμε,
- και η ερμηνεία δεν βασίζεται σε post-hoc selection ή cherry-picking.

### Controlled ablation dimensions

Στο παρόν notebook θα εξεταστούν μόνο τρεις strict διαστάσεις:

1. **Spatial information control**
   - `self_only_control`
   - χωρίς neighbor message passing πέρα από self information

2. **Graph topology choice**
   - `canonical_nb11_graph`
   - `local_pruned_graph`
   - deterministic pruning του canonical graph ώστε να κρατηθούν μόνο πιο τοπικές συνδέσεις

3. **Aggregation depth**
   - `1` layer
   - `2` layers

### Scope discipline

Το registry:

- **δεν** ανοίγει νέο broad benchmark scope,
- **δεν** εισάγει νέα model families,
- **δεν** αλλάζει το benchmark authority,
- και **δεν** χρησιμοποιεί το test split για model selection.

### Evaluation rule

Για όλα τα runs του `NB13`:

- το **validation loss** χρησιμοποιείται για model selection / early stopping,
- το **test split** χρησιμοποιείται μόνο για final reporting,
- και τα canonical benchmark references παραμένουν read-only.

In [4]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from src import config as cfg


# ============================================================
# 1. Παίρνουμε το reference training setup από το NB12
#    και το παγώνουμε ως fixed configuration για το NB13.
#    Έτσι αλλάζουν μόνο οι ablation dimensions.
# ============================================================
nb12_training_cfg = dict(nb12_run_config["training_config"])

required_training_keys = [
    "batch_size",
    "hidden_channels",
    "dropout",
    "learning_rate",
    "weight_decay",
    "max_epochs",
    "patience",
    "min_delta",
    "gradient_clip_norm",
]

missing_training_keys = [
    key for key in required_training_keys if key not in nb12_training_cfg
]
if missing_training_keys:
    raise KeyError(
        f"[NB12 TRAINING CONFIG INCOMPLETE] missing keys: {missing_training_keys}"
    )


# ============================================================
# 2. Δηλώνουμε εκ των προτέρων το NB13 experiment registry.
#    Δεν κάνουμε ad hoc πειράματα μετά το training.
# ============================================================
# Scope-safe design:
# - self_only_control: no neighbor message passing
# - canonical_nb11_graph: original graph baseline topology
# - local_pruned_graph: deterministic short-range pruning of canonical edges
#
# Controlled factors:
# - topology_variant
# - message_passing_layers
#
# Frozen factors:
# - hidden size
# - dropout
# - optimizer settings
# - early stopping settings
# - batch size
#
# Primary research questions:
# - Does spatial information help at all?
# - Is performance sensitive to topology choice?
# - Does deeper aggregation degrade performance?

registry_rows = [
    {
        "experiment_id": "NB13_E01",
        "enabled": True,
        "topology_variant": "self_only_control",
        "message_passing_layers": 1,
        "topology_source": "identity_self_loop_only",
        "edge_policy": "self_loops_only",
        "edge_keep_quantile": None,
        "uses_neighbor_messages": False,
        "selection_metric": "val_loss",
        "reporting_split": "test",
        "notes": "No-spatial control for direct comparison against graph variants.",
    },
    {
        "experiment_id": "NB13_E02",
        "enabled": True,
        "topology_variant": "self_only_control",
        "message_passing_layers": 2,
        "topology_source": "identity_self_loop_only",
        "edge_policy": "self_loops_only",
        "edge_keep_quantile": None,
        "uses_neighbor_messages": False,
        "selection_metric": "val_loss",
        "reporting_split": "test",
        "notes": "Depth control without neighbor aggregation.",
    },
    {
        "experiment_id": "NB13_E03",
        "enabled": True,
        "topology_variant": "canonical_nb11_graph",
        "message_passing_layers": 1,
        "topology_source": "nb11_packaged_graph",
        "edge_policy": "use_all_canonical_edges",
        "edge_keep_quantile": 1.0,
        "uses_neighbor_messages": True,
        "selection_metric": "val_loss",
        "reporting_split": "test",
        "notes": "Canonical graph baseline at shallow aggregation depth.",
    },
    {
        "experiment_id": "NB13_E04",
        "enabled": True,
        "topology_variant": "canonical_nb11_graph",
        "message_passing_layers": 2,
        "topology_source": "nb11_packaged_graph",
        "edge_policy": "use_all_canonical_edges",
        "edge_keep_quantile": 1.0,
        "uses_neighbor_messages": True,
        "selection_metric": "val_loss",
        "reporting_split": "test",
        "notes": "Canonical graph baseline at deeper aggregation depth.",
    },
    {
        "experiment_id": "NB13_E05",
        "enabled": True,
        "topology_variant": "local_pruned_graph",
        "message_passing_layers": 1,
        "topology_source": "canonical_graph_edge_pruning",
        "edge_policy": "keep_shortest_edges_only",
        "edge_keep_quantile": 0.50,
        "uses_neighbor_messages": True,
        "selection_metric": "val_loss",
        "reporting_split": "test",
        "notes": "Local-only graph variant using deterministic short-edge pruning.",
    },
    {
        "experiment_id": "NB13_E06",
        "enabled": True,
        "topology_variant": "local_pruned_graph",
        "message_passing_layers": 2,
        "topology_source": "canonical_graph_edge_pruning",
        "edge_policy": "keep_shortest_edges_only",
        "edge_keep_quantile": 0.50,
        "uses_neighbor_messages": True,
        "selection_metric": "val_loss",
        "reporting_split": "test",
        "notes": "Local-only graph variant with deeper aggregation.",
    },
]

experiment_registry_df = pd.DataFrame(registry_rows)

# Προσθέτουμε σε κάθε experiment το ίδιο fixed training setup του NB12.
for key, value in nb12_training_cfg.items():
    experiment_registry_df[key] = value

# Canonical metadata για reproducibility και downstream reporting.
experiment_registry_df["seed"] = int(nb12_run_config["seed"])
experiment_registry_df["device_reference"] = str(nb12_run_config["device"])
experiment_registry_df["n_train_snapshots"] = len(train_dataset)
experiment_registry_df["n_val_snapshots"] = len(val_dataset)
experiment_registry_df["n_test_snapshots"] = len(test_dataset)
experiment_registry_df["n_nodes"] = expected_n_nodes
experiment_registry_df["input_feature_dim"] = train_dataset.n_input_features
experiment_registry_df["reference_nb12_best_epoch"] = int(nb12_run_config["best_epoch"])
experiment_registry_df["reference_nb12_best_val_loss"] = float(nb12_run_config["best_val_loss"])
experiment_registry_df["benchmark_reference_mode"] = "read_only"
experiment_registry_df["status"] = "planned"


# ============================================================
# 3. Fail-fast validation του registry.
#    Αν κάτι είναι inconsistent, το notebook πρέπει να σπάσει εδώ
#    και όχι αφού ξεκινήσει training.
# ============================================================
if experiment_registry_df["experiment_id"].duplicated().any():
    duplicated_ids = experiment_registry_df.loc[
        experiment_registry_df["experiment_id"].duplicated(),
        "experiment_id",
    ].tolist()
    raise ValueError(f"[DUPLICATE EXPERIMENT IDS] {duplicated_ids}")

allowed_topology_variants = {
    "self_only_control",
    "canonical_nb11_graph",
    "local_pruned_graph",
}
unexpected_topologies = sorted(
    set(experiment_registry_df["topology_variant"]) - allowed_topology_variants
)
if unexpected_topologies:
    raise ValueError(f"[UNEXPECTED TOPOLOGY VARIANTS] {unexpected_topologies}")

allowed_layer_values = {1, 2}
unexpected_layers = sorted(
    set(experiment_registry_df["message_passing_layers"]) - allowed_layer_values
)
if unexpected_layers:
    raise ValueError(f"[UNEXPECTED LAYER COUNTS] {unexpected_layers}")

if not experiment_registry_df["enabled"].any():
    raise ValueError("[EMPTY RUN PLAN] No enabled experiments in NB13 registry.")

# Ο self-only control δεν επιτρέπεται να χρησιμοποιεί neighbor messages.
control_rows = experiment_registry_df[
    experiment_registry_df["topology_variant"] == "self_only_control"
]
if control_rows["uses_neighbor_messages"].any():
    raise ValueError("[INVALID CONTROL] self_only_control must disable neighbor messages.")

# Το canonical topology κρατά όλα τα canonical edges.
canonical_rows = experiment_registry_df[
    experiment_registry_df["topology_variant"] == "canonical_nb11_graph"
]
if not (canonical_rows["edge_keep_quantile"] == 1.0).all():
    raise ValueError("[INVALID CANONICAL TOPOLOGY] canonical graph must keep all edges.")

# Το local-pruned topology πρέπει όντως να κάνει pruning.
local_rows = experiment_registry_df[
    experiment_registry_df["topology_variant"] == "local_pruned_graph"
]
if not (local_rows["edge_keep_quantile"] < 1.0).all():
    raise ValueError("[INVALID LOCAL PRUNING] local_pruned_graph must prune edges.")

# Κλειδώνουμε ρητά τον evaluation κανόνα του NB13.
if not (experiment_registry_df["selection_metric"] == "val_loss").all():
    raise ValueError("[INVALID SELECTION RULE] all runs must select on validation loss.")

if not (experiment_registry_df["reporting_split"] == "test").all():
    raise ValueError("[INVALID REPORTING RULE] final reporting must be test-only.")

if not (experiment_registry_df["benchmark_reference_mode"] == "read_only").all():
    raise ValueError("[INVALID BENCHMARK MODE] benchmark references must remain read-only.")


# ============================================================
# 4. Αποθηκεύουμε:
#    - το experiment registry
#    - ένα μικρό protocol-lock JSON
#    ώστε το notebook να έχει explicit run contract.
# ============================================================
Path(cfg.NB13_GRAPH_ABLATION_DIR).mkdir(parents=True, exist_ok=True)

experiment_registry_df.to_csv(cfg.NB13_EXPERIMENT_REGISTRY, index=False)

nb13_protocol_lock = {
    "notebook": "13_graph_ablation_and_spatial_sensitivity_analysis.ipynb",
    "role": "strict_follow_up_of_nb12",
    "scope": "graph_ablation_and_spatial_sensitivity",
    "selection_rule": "validation_only_for_model_selection",
    "reporting_rule": "test_only_for_final_reporting",
    "benchmark_reference_mode": "read_only",
    "reference_nb12_run_config": {
        "seed": int(nb12_run_config["seed"]),
        "device": nb12_run_config["device"],
        "best_epoch": int(nb12_run_config["best_epoch"]),
        "best_val_loss": float(nb12_run_config["best_val_loss"]),
        "training_config": nb12_training_cfg,
    },
    "frozen_dimensions": [
        "seed",
        "hidden_channels",
        "dropout",
        "learning_rate",
        "weight_decay",
        "max_epochs",
        "patience",
        "min_delta",
        "gradient_clip_norm",
        "batch_size",
        "input_feature_dim",
        "n_nodes",
    ],
    "ablation_dimensions": [
        "topology_variant",
        "edge_policy",
        "edge_keep_quantile",
        "message_passing_layers",
    ],
    "planned_experiments": experiment_registry_df["experiment_id"].tolist(),
    "notes": [
        "No new model family is introduced in NB13.",
        "NB13 is forecasting-first and non-overclaiming.",
        "The goal is sensitivity analysis, not graph superiority claims.",
    ],
}

with open(cfg.NB13_RUN_CONFIG, "w", encoding="utf-8") as f:
    json.dump(nb13_protocol_lock, f, ensure_ascii=False, indent=2)


# ============================================================
# 5. Προβάλλουμε το frozen registry που θα οδηγήσει τα runs.
# ============================================================
display_cols = [
    "experiment_id",
    "enabled",
    "topology_variant",
    "message_passing_layers",
    "edge_policy",
    "edge_keep_quantile",
    "uses_neighbor_messages",
    "hidden_channels",
    "dropout",
    "learning_rate",
    "weight_decay",
    "batch_size",
    "max_epochs",
    "patience",
    "status",
]

display(experiment_registry_df[display_cols])

print("NB13 experiment registry locked successfully.")
print(f"Saved registry : {cfg.NB13_EXPERIMENT_REGISTRY}")
print(f"Saved protocol : {cfg.NB13_RUN_CONFIG}")

,experiment_id,enabled,topology_variant,message_passing_layers,edge_policy,edge_keep_quantile,uses_neighbor_messages,hidden_channels,dropout,learning_rate,weight_decay,batch_size,max_epochs,patience,status
0,NB13_E01,True,self_only_control,1,self_loops_only,NaN,False,64,0.2,0.001,0.00001,16,30,6,planned
1,NB13_E02,True,self_only_control,2,self_loops_only,NaN,False,64,0.2,0.001,0.00001,16,30,6,planned
2,NB13_E03,True,canonical_nb11_graph,1,use_all_canonical_edges,1.0,True,64,0.2,0.001,0.00001,16,30,6,planned
3,NB13_E04,True,canonical_nb11_graph,2,use_all_canonical_edges,1.0,True,64,0.2,0.001,0.00001,16,30,6,planned
4,NB13_E05,True,local_pruned_graph,1,keep_shortest_edges_only,0.5,True,64,0.2,0.001,0.00001,16,30,6,planned
5,NB13_E06,True,local_pruned_graph,2,keep_shortest_edges_only,0.5,True,64,0.2,0.001,0.00001,16,30,6,planned


NB13 experiment registry locked successfully.
Saved registry : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_experiment_registry.csv
Saved protocol : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_run_config.json


## 4. Deterministic topology builders και edge-transformation helpers

Σε αυτό το section ορίζονται οι graph topology variants του `NB13`
με deterministic και benchmark-safe τρόπο.

### Topology variants

1. **`self_only_control`**
   - κάθε node διατηρεί μόνο self-loop
   - δεν υπάρχει spatial neighbor message passing

2. **`canonical_nb11_graph`**
   - χρησιμοποιείται το canonical graph όπως παραδόθηκε από το `NB11`

3. **`local_pruned_graph`**
   - χρησιμοποιείται deterministic pruning του canonical graph
   - κρατούνται μόνο οι πιο τοπικές συνδέσεις με βάση το edge distance

### Methodological constraint

Το section αυτό:

- **δεν** ξανακατασκευάζει νέο graph από raw coordinates,
- **δεν** αλλάζει node ordering,
- **δεν** αλλάζει feature space,
- και **δεν** αλλάζει split semantics.

Αντίθετα, εφαρμόζει controlled edge transformations πάνω στο ήδη verified graph contract.

In [5]:
import copy
import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. Βασικά helpers για edge-bundle validation
# ============================================================
def assert_edge_bundle_consistency(
    edge_index: torch.Tensor,
    edge_attr: torch.Tensor,
    label: str,
) -> None:
    """Fail-fast έλεγχος ότι edge_index / edge_attr είναι συνεπή."""
    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError(f"[INVALID EDGE_INDEX] {label}: expected shape (2, E)")

    if edge_attr.ndim != 2:
        raise ValueError(f"[INVALID EDGE_ATTR] {label}: expected shape (E, A)")

    if edge_index.shape[1] != edge_attr.shape[0]:
        raise ValueError(
            f"[EDGE BUNDLE MISMATCH] {label}: "
            f"edge_index has {edge_index.shape[1]} edges but edge_attr has {edge_attr.shape[0]} rows"
        )

    if edge_index.shape[1] <= 0:
        raise ValueError(f"[EMPTY EDGE BUNDLE] {label}")


def count_self_loops(edge_index: torch.Tensor) -> int:
    return int((edge_index[0] == edge_index[1]).sum().item())


def edge_bundle_summary(
    edge_index: torch.Tensor,
    edge_attr: torch.Tensor,
    label: str,
) -> dict:
    """Σύνοψη topology για notebook-level verification."""
    assert_edge_bundle_consistency(edge_index, edge_attr, label)

    src = edge_index[0].cpu().numpy()
    dst = edge_index[1].cpu().numpy()
    dist = edge_attr[:, 0].cpu().numpy()

    undirected_pairs = {
        tuple(sorted((int(u), int(v))))
        for u, v in zip(src, dst)
        if int(u) != int(v)
    }

    non_self_dist = dist[src != dst]

    return {
        "topology_label": label,
        "directed_edges": int(edge_index.shape[1]),
        "unique_undirected_edges": int(len(undirected_pairs)),
        "self_loops": count_self_loops(edge_index),
        "edge_attr_dim": int(edge_attr.shape[1]),
        "min_distance_km_nonself": float(non_self_dist.min()) if len(non_self_dist) else 0.0,
        "max_distance_km_nonself": float(non_self_dist.max()) if len(non_self_dist) else 0.0,
    }


# ============================================================
# 2. Επιβεβαιώνουμε ότι το canonical graph είναι ίδιο σε όλα τα splits
# ============================================================
def tensors_equal(a: torch.Tensor, b: torch.Tensor) -> bool:
    return a.shape == b.shape and torch.equal(a.cpu(), b.cpu())


if not tensors_equal(train_package["edge_index"], val_package["edge_index"]):
    raise ValueError("[CANONICAL GRAPH MISMATCH] train and val edge_index differ.")

if not tensors_equal(train_package["edge_index"], test_package["edge_index"]):
    raise ValueError("[CANONICAL GRAPH MISMATCH] train and test edge_index differ.")

if not tensors_equal(train_package["edge_attr_km"], val_package["edge_attr_km"]):
    raise ValueError("[CANONICAL GRAPH MISMATCH] train and val edge_attr_km differ.")

if not tensors_equal(train_package["edge_attr_km"], test_package["edge_attr_km"]):
    raise ValueError("[CANONICAL GRAPH MISMATCH] train and test edge_attr_km differ.")


canonical_edge_index = train_package["edge_index"].clone().cpu().long()
canonical_edge_attr_km = train_package["edge_attr_km"].clone().cpu().float()

assert_edge_bundle_consistency(
    canonical_edge_index,
    canonical_edge_attr_km,
    "canonical_nb11_graph",
)


# ============================================================
# 3. Builders για τις topology variants
# ============================================================
def build_self_only_graph(n_nodes: int) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Χτίζει topology μόνο με self-loops.
    Το edge_attr κρατιέται ως single-column zero distance για shape consistency.
    """
    node_idx = torch.arange(n_nodes, dtype=torch.long)
    edge_index = torch.stack([node_idx, node_idx], dim=0)
    edge_attr_km = torch.zeros((n_nodes, 1), dtype=torch.float32)

    assert_edge_bundle_consistency(edge_index, edge_attr_km, "self_only_control")
    return edge_index, edge_attr_km


def canonical_edge_dataframe(
    edge_index: torch.Tensor,
    edge_attr_km: torch.Tensor,
) -> pd.DataFrame:
    """
    Μετατρέπει το directed canonical graph σε undirected dataframe.
    Για κάθε ζεύγος nodes κρατάμε ένα μόνο row με το distance_km.
    """
    src = edge_index[0].cpu().numpy().astype(int)
    dst = edge_index[1].cpu().numpy().astype(int)
    dist = edge_attr_km[:, 0].cpu().numpy().astype(float)

    rows = []
    for u, v, d in zip(src, dst, dist):
        if u == v:
            # Τα self-loops δεν συμμετέχουν στο pruning του local graph
            continue

        a, b = sorted((int(u), int(v)))
        rows.append((a, b, float(d)))

    df = pd.DataFrame(rows, columns=["u", "v", "distance_km"])
    if df.empty:
        raise ValueError("[EMPTY CANONICAL EDGE DF] No non-self canonical edges found.")

    # Κρατάμε ένα μοναδικό undirected pair με deterministic aggregation
    df = (
        df.groupby(["u", "v"], as_index=False)["distance_km"]
        .min()
        .sort_values(["distance_km", "u", "v"], ascending=[True, True, True])
        .reset_index(drop=True)
    )
    return df


def build_local_pruned_graph(
    canonical_edge_index: torch.Tensor,
    canonical_edge_attr_km: torch.Tensor,
    keep_quantile: float,
) -> tuple[torch.Tensor, torch.Tensor, pd.DataFrame]:
    """
    Κρατά deterministic subset των shortest undirected edges και
    μετά ανακατασκευάζει bidirectional directed graph.
    """
    if not (0.0 < float(keep_quantile) < 1.0):
        raise ValueError(
            f"[INVALID KEEP QUANTILE] Expected 0 < q < 1 for pruning, got {keep_quantile}"
        )

    undirected_df = canonical_edge_dataframe(
        canonical_edge_index,
        canonical_edge_attr_km,
    )

    distance_threshold = float(undirected_df["distance_km"].quantile(keep_quantile))
    kept_df = (
        undirected_df.loc[undirected_df["distance_km"] <= distance_threshold]
        .copy()
        .sort_values(["distance_km", "u", "v"], ascending=[True, True, True])
        .reset_index(drop=True)
    )

    if kept_df.empty:
        raise ValueError("[EMPTY PRUNED GRAPH] No edges remained after pruning.")

    directed_edges = []
    directed_attr = []

    # Ξανακάνουμε expand σε bidirectional directed edges
    for row in kept_df.itertuples(index=False):
        u, v, d = int(row.u), int(row.v), float(row.distance_km)
        directed_edges.append((u, v))
        directed_edges.append((v, u))
        directed_attr.append([d])
        directed_attr.append([d])

    edge_index = torch.tensor(directed_edges, dtype=torch.long).T.contiguous()
    edge_attr_km = torch.tensor(directed_attr, dtype=torch.float32)

    assert_edge_bundle_consistency(edge_index, edge_attr_km, "local_pruned_graph")
    return edge_index, edge_attr_km, kept_df


def replace_graph_in_package(
    package: dict,
    new_edge_index: torch.Tensor,
    new_edge_attr_km: torch.Tensor,
) -> dict:
    """
    Επιστρέφει shallow copy του package με νέο graph topology,
    χωρίς να αλλάζει τίποτε άλλο στο split package.
    """
    new_package = dict(package)
    new_package["edge_index"] = new_edge_index.clone()
    new_package["edge_attr_km"] = new_edge_attr_km.clone()
    return new_package


# ============================================================
# 4. Χτίζουμε όλες τις topology variants μία φορά, deterministic
# ============================================================
self_only_edge_index, self_only_edge_attr_km = build_self_only_graph(expected_n_nodes)

local_keep_quantile = float(
    experiment_registry_df.loc[
        experiment_registry_df["topology_variant"] == "local_pruned_graph",
        "edge_keep_quantile",
    ].dropna().iloc[0]
)

local_pruned_edge_index, local_pruned_edge_attr_km, local_pruned_undirected_df = (
    build_local_pruned_graph(
        canonical_edge_index=canonical_edge_index,
        canonical_edge_attr_km=canonical_edge_attr_km,
        keep_quantile=local_keep_quantile,
    )
)

topology_bundles = {
    "self_only_control": {
        "edge_index": self_only_edge_index,
        "edge_attr_km": self_only_edge_attr_km,
    },
    "canonical_nb11_graph": {
        "edge_index": canonical_edge_index,
        "edge_attr_km": canonical_edge_attr_km,
    },
    "local_pruned_graph": {
        "edge_index": local_pruned_edge_index,
        "edge_attr_km": local_pruned_edge_attr_km,
    },
}


# ============================================================
# 5. Χτίζουμε topology-specific packages για train / val / test
# ============================================================
topology_packages = {}

for topology_name, bundle in topology_bundles.items():
    topology_packages[topology_name] = {
        "train": replace_graph_in_package(
            train_package,
            bundle["edge_index"],
            bundle["edge_attr_km"],
        ),
        "val": replace_graph_in_package(
            val_package,
            bundle["edge_index"],
            bundle["edge_attr_km"],
        ),
        "test": replace_graph_in_package(
            test_package,
            bundle["edge_index"],
            bundle["edge_attr_km"],
        ),
    }


# ============================================================
# 6. Verification summary για όλες τις topology variants
# ============================================================
topology_summary_rows = []

for topology_name, bundle in topology_bundles.items():
    row = edge_bundle_summary(
        bundle["edge_index"],
        bundle["edge_attr_km"],
        topology_name,
    )

    if topology_name == "local_pruned_graph":
        row["edge_keep_quantile"] = local_keep_quantile
        row["kept_undirected_edges_after_pruning"] = int(len(local_pruned_undirected_df))
    else:
        row["edge_keep_quantile"] = np.nan
        row["kept_undirected_edges_after_pruning"] = np.nan

    topology_summary_rows.append(row)

topology_summary_df = pd.DataFrame(topology_summary_rows)
display(topology_summary_df)

print("Topology builders ready.")
print("Available topology packages:", list(topology_packages.keys()))

,topology_label,directed_edges,unique_undirected_edges,self_loops,edge_attr_dim,min_distance_km_nonself,max_distance_km_nonself,edge_keep_quantile,kept_undirected_edges_after_pruning
0,self_only_control,256,0,256,1,0.000000,0.000000,NaN,NaN
1,canonical_nb11_graph,1068,534,0,1,4.426395,49.962254,NaN,NaN
2,local_pruned_graph,534,267,0,1,4.426395,38.750343,0.5,267.0


Topology builders ready.
Available topology packages: ['self_only_control', 'canonical_nb11_graph', 'local_pruned_graph']


## 5. Topology-aware snapshot dataset materialization και DataLoader helpers

Σε αυτό το section μετατρέπουμε τα topology-specific packaged splits
σε actual snapshot-level datasets και DataLoaders.

### Στόχος

Για κάθε topology variant του `NB13` θέλουμε να κατασκευάσουμε:

- `train_dataset`
- `val_dataset`
- `test_dataset`
- `train_loader`
- `val_loader`
- `test_loader`

ώστε κάθε planned experiment του registry να μπορεί να εκτελεστεί
πάνω σε πλήρως deterministic input bundle.

### Methodological note

Το section αυτό:

- **δεν** αλλάζει targets ή features,
- **δεν** αλλάζει observed-mask logic,
- **δεν** αλλάζει split membership,
- και **δεν** κάνει model selection.

Απλώς υλοποιεί το runtime handoff από:

`topology-specific package -> snapshot dataset -> DataLoader`

με benchmark-safe και reproducible τρόπο.

In [6]:
from pathlib import Path

import pandas as pd
from torch_geometric.loader import DataLoader


# ============================================================
# 1. Runtime constants για DataLoaders
# ============================================================
# Κρατάμε το batch size από το frozen NB12-compatible registry.
unique_batch_sizes = experiment_registry_df["batch_size"].dropna().unique().tolist()
if len(unique_batch_sizes) != 1:
    raise ValueError(
        f"[AMBIGUOUS BATCH SIZE] Expected one frozen batch size, got {unique_batch_sizes}"
    )

runtime_batch_size = int(unique_batch_sizes[0])

# Reproducibility-friendly loader policy:
# - shuffle μόνο στο train
# - val / test deterministic
# - num_workers=0 για cross-platform notebook stability
LOADER_KWARGS = {
    "batch_size": runtime_batch_size,
    "num_workers": 0,
    "pin_memory": False,
    "drop_last": False,
}


# ============================================================
# 2. Helpers για dataset / loader materialization
# ============================================================
def build_split_datasets_from_packages(
    split_packages: dict,
    include_observed_indicator: bool = True,
) -> dict:
    """
    Δημιουργεί snapshot datasets για train / val / test
    από topology-specific packaged splits.
    """
    required_split_names = {"train", "val", "test"}
    if set(split_packages.keys()) != required_split_names:
        raise ValueError(
            f"[INVALID SPLIT PACKAGE BUNDLE] Expected keys {required_split_names}, "
            f"got {set(split_packages.keys())}"
        )

    datasets = {
        "train": TemporalGraphSnapshotDataset(
            split_packages["train"],
            include_observed_indicator=include_observed_indicator,
        ),
        "val": TemporalGraphSnapshotDataset(
            split_packages["val"],
            include_observed_indicator=include_observed_indicator,
        ),
        "test": TemporalGraphSnapshotDataset(
            split_packages["test"],
            include_observed_indicator=include_observed_indicator,
        ),
    }

    return datasets


def build_split_loaders_from_datasets(
    split_datasets: dict,
    loader_kwargs: dict,
) -> dict:
    """
    Δημιουργεί DataLoaders για τα ήδη materialized datasets.
    """
    required_split_names = {"train", "val", "test"}
    if set(split_datasets.keys()) != required_split_names:
        raise ValueError(
            f"[INVALID SPLIT DATASET BUNDLE] Expected keys {required_split_names}, "
            f"got {set(split_datasets.keys())}"
        )

    train_loader = DataLoader(
        split_datasets["train"],
        shuffle=True,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        split_datasets["val"],
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        split_datasets["test"],
        shuffle=False,
        **loader_kwargs,
    )

    return {
        "train": train_loader,
        "val": val_loader,
        "test": test_loader,
    }


def summarize_runtime_bundle(
    topology_name: str,
    split_datasets: dict,
    split_loaders: dict,
) -> dict:
    """
    Συνοψίζει το runtime bundle ενός topology για notebook-level verification.
    """
    train_sample = split_datasets["train"][0]
    val_sample = split_datasets["val"][0]
    test_sample = split_datasets["test"][0]

    return {
        "topology_variant": topology_name,
        "batch_size": runtime_batch_size,
        "train_dataset_len": len(split_datasets["train"]),
        "val_dataset_len": len(split_datasets["val"]),
        "test_dataset_len": len(split_datasets["test"]),
        "train_batches": len(split_loaders["train"]),
        "val_batches": len(split_loaders["val"]),
        "test_batches": len(split_loaders["test"]),
        "train_first_x_shape": str(tuple(train_sample.x.shape)),
        "val_first_x_shape": str(tuple(val_sample.x.shape)),
        "test_first_x_shape": str(tuple(test_sample.x.shape)),
        "train_first_y_shape": str(tuple(train_sample.y.shape)),
        "val_first_y_shape": str(tuple(val_sample.y.shape)),
        "test_first_y_shape": str(tuple(test_sample.y.shape)),
    }


# ============================================================
# 3. Materialize topology-specific datasets and loaders
# ============================================================
topology_runtime_bundles = {}
runtime_summary_rows = []

for topology_name, split_packages in topology_packages.items():
    split_datasets = build_split_datasets_from_packages(
        split_packages=split_packages,
        include_observed_indicator=True,
    )
    split_loaders = build_split_loaders_from_datasets(
        split_datasets=split_datasets,
        loader_kwargs=LOADER_KWARGS,
    )

    # Fail-fast reconciliation με το ήδη verified canonical split sizes
    if len(split_datasets["train"]) != len(train_dataset):
        raise ValueError(
            f"[TRAIN LENGTH MISMATCH] {topology_name}: "
            f"{len(split_datasets['train'])} vs {len(train_dataset)}"
        )
    if len(split_datasets["val"]) != len(val_dataset):
        raise ValueError(
            f"[VAL LENGTH MISMATCH] {topology_name}: "
            f"{len(split_datasets['val'])} vs {len(val_dataset)}"
        )
    if len(split_datasets["test"]) != len(test_dataset):
        raise ValueError(
            f"[TEST LENGTH MISMATCH] {topology_name}: "
            f"{len(split_datasets['test'])} vs {len(test_dataset)}"
        )

    # Ελέγχουμε ότι το input feature dimension παραμένει ακριβώς το ίδιο.
    if split_datasets["train"].n_input_features != train_dataset.n_input_features:
        raise ValueError(
            f"[FEATURE DIM MISMATCH] {topology_name}: "
            f"{split_datasets['train'].n_input_features} vs {train_dataset.n_input_features}"
        )

    topology_runtime_bundles[topology_name] = {
        "datasets": split_datasets,
        "loaders": split_loaders,
    }

    runtime_summary_rows.append(
        summarize_runtime_bundle(
            topology_name=topology_name,
            split_datasets=split_datasets,
            split_loaders=split_loaders,
        )
    )

runtime_summary_df = pd.DataFrame(runtime_summary_rows)
display(runtime_summary_df)


# ============================================================
# 4. Build experiment-level runtime map from the frozen registry
# ============================================================
experiment_runtime_map = {}

for row in experiment_registry_df.itertuples(index=False):
    topology_name = row.topology_variant

    if topology_name not in topology_runtime_bundles:
        raise KeyError(
            f"[MISSING TOPOLOGY RUNTIME BUNDLE] experiment_id={row.experiment_id}, "
            f"topology_variant={topology_name}"
        )

    experiment_runtime_map[row.experiment_id] = {
        "topology_variant": topology_name,
        "message_passing_layers": int(row.message_passing_layers),
        "uses_neighbor_messages": bool(row.uses_neighbor_messages),
        "datasets": topology_runtime_bundles[topology_name]["datasets"],
        "loaders": topology_runtime_bundles[topology_name]["loaders"],
    }


# ============================================================
# 5. Export ένα μικρό materialization summary για traceability
# ============================================================
nb13_dataset_materialization_summary_path = (
    Path(cfg.NB13_GRAPH_ABLATION_DIR) / "nb13_dataset_materialization_summary.csv"
)
runtime_summary_df.to_csv(nb13_dataset_materialization_summary_path, index=False)


# ============================================================
# 6. Final status
# ============================================================
print("Topology-aware dataset materialization completed successfully.")
print(f"Batch size used           : {runtime_batch_size}")
print(f"Topology runtime bundles  : {list(topology_runtime_bundles.keys())}")
print(f"Experiment runtime map    : {list(experiment_runtime_map.keys())}")
print(f"Saved runtime summary     : {nb13_dataset_materialization_summary_path}")

,topology_variant,batch_size,train_dataset_len,val_dataset_len,test_dataset_len,train_batches,val_batches,test_batches,train_first_x_shape,val_first_x_shape,test_first_x_shape,train_first_y_shape,val_first_y_shape,test_first_y_shape
0,self_only_control,16,7819,720,4295,489,45,269,"(256, 42)","(256, 42)","(256, 42)","(256,)","(256,)","(256,)"
1,canonical_nb11_graph,16,7819,720,4295,489,45,269,"(256, 42)","(256, 42)","(256, 42)","(256,)","(256,)","(256,)"
2,local_pruned_graph,16,7819,720,4295,489,45,269,"(256, 42)","(256, 42)","(256, 42)","(256,)","(256,)","(256,)"


Topology-aware dataset materialization completed successfully.
Batch size used           : 16
Topology runtime bundles  : ['self_only_control', 'canonical_nb11_graph', 'local_pruned_graph']
Experiment runtime map    : ['NB13_E01', 'NB13_E02', 'NB13_E03', 'NB13_E04', 'NB13_E05', 'NB13_E06']
Saved runtime summary     : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_dataset_materialization_summary.csv


## 6. GCN model definition, masked loss helpers και experiment runner skeleton

Σε αυτό το section ορίζεται το canonical execution backbone του `NB13`.

### Τι περιλαμβάνει

- ένα μικρό και controlled `GCN` regressor,
- masked regression loss πάνω στο `observed_mask`,
- train / validation / test evaluation helpers,
- και ένα early-stopping-ready experiment runner skeleton.

### Methodological intent

Το `NB13` δεν εισάγει νέα model family.
Αντίθετα, κρατά το ίδιο γενικό modeling family με το `NB12`
και εξετάζει μόνο controlled αλλαγές σε:

- graph topology,
- και aggregation depth.

### Evaluation discipline

Η υλοποίηση παραμένει αυστηρά:

- validation-driven για model selection,
- test-only για final reporting,
- benchmark-safe,
- και non-overclaiming.

In [7]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


# ============================================================
# 1. Controlled GCN regressor για το NB13
# ============================================================
class GCNForecastRegressor(nn.Module):
    """
    Απλό και controlled GCN backbone για graph-based forecasting.

    Το NB13 δεν αλλάζει model family.
    Εξετάζει μόνο topology και aggregation depth.
    """

    def __init__(
        self,
        input_dim: int,
        hidden_channels: int,
        dropout: float,
        message_passing_layers: int,
    ) -> None:
        super().__init__()

        if message_passing_layers not in {1, 2}:
            raise ValueError(
                f"Unsupported message_passing_layers={message_passing_layers}. "
                "NB13 only supports {1, 2}."
            )

        self.input_dim = int(input_dim)
        self.hidden_channels = int(hidden_channels)
        self.dropout = float(dropout)
        self.message_passing_layers = int(message_passing_layers)

        self.conv1 = GCNConv(self.input_dim, self.hidden_channels)

        if self.message_passing_layers == 2:
            self.conv2 = GCNConv(self.hidden_channels, self.hidden_channels)
        else:
            self.conv2 = None

        self.output_layer = nn.Linear(self.hidden_channels, 1)

    def forward(self, batch) -> torch.Tensor:
        x = batch.x
        edge_index = batch.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        if self.conv2 is not None:
            x = self.conv2(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        out = self.output_layer(x).squeeze(-1)
        return out


# ============================================================
# 2. Masked loss / metric helpers
# ============================================================
def ensure_boolean_mask(mask: torch.Tensor) -> torch.Tensor:
    if mask.dtype == torch.bool:
        return mask
    return mask > 0


def get_masked_vectors(
    prediction: torch.Tensor,
    target: torch.Tensor,
    observed_mask: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Κρατά μόνο τα observed entries του current batch.
    """
    mask = ensure_boolean_mask(observed_mask)

    pred_obs = prediction[mask]
    target_obs = target[mask]

    if pred_obs.numel() == 0:
        raise ValueError("[EMPTY OBSERVED VECTOR] No observed entries available.")

    return pred_obs, target_obs


def masked_mse_loss(
    prediction: torch.Tensor,
    target: torch.Tensor,
    observed_mask: torch.Tensor,
) -> torch.Tensor:
    pred_obs, target_obs = get_masked_vectors(prediction, target, observed_mask)
    return F.mse_loss(pred_obs, target_obs)


def masked_mae_value(
    prediction: torch.Tensor,
    target: torch.Tensor,
    observed_mask: torch.Tensor,
) -> float:
    pred_obs, target_obs = get_masked_vectors(prediction, target, observed_mask)
    return float(torch.mean(torch.abs(pred_obs - target_obs)).item())


def masked_rmse_value(
    prediction: torch.Tensor,
    target: torch.Tensor,
    observed_mask: torch.Tensor,
) -> float:
    pred_obs, target_obs = get_masked_vectors(prediction, target, observed_mask)
    return float(torch.sqrt(torch.mean((pred_obs - target_obs) ** 2)).item())


def compute_r2_from_numpy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    R² πάνω στο observed evaluation space.
    """
    if y_true.size == 0:
        return float("nan")

    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))

    if ss_tot <= 0.0:
        return float("nan")

    return float(1.0 - (ss_res / ss_tot))


# ============================================================
# 3. Early stopping helper
# ============================================================
@dataclass
class EarlyStoppingState:
    best_epoch: int | None = None
    best_val_loss: float = float("inf")
    epochs_without_improvement: int = 0
    best_model_state: dict | None = None

    def update(
        self,
        epoch: int,
        current_val_loss: float,
        model: nn.Module,
        min_delta: float,
    ) -> bool:
        """
        Επιστρέφει True αν υπάρχει ουσιαστική βελτίωση.
        """
        improvement = self.best_val_loss - float(current_val_loss)

        if improvement > float(min_delta):
            self.best_epoch = int(epoch)
            self.best_val_loss = float(current_val_loss)
            self.epochs_without_improvement = 0
            self.best_model_state = deepcopy(model.state_dict())
            return True

        self.epochs_without_improvement += 1
        return False


# ============================================================
# 4. Train / evaluation helpers
# ============================================================
def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    gradient_clip_norm: float,
) -> dict:
    """
    Ένα training epoch με observed-mask-aware aggregation.
    """
    model.train()

    total_squared_error = 0.0
    total_absolute_error = 0.0
    total_observed_count = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        prediction = model(batch)
        loss = masked_mse_loss(prediction, batch.y, batch.observed_mask)

        loss.backward()

        # Gradient clipping για σταθερότητα, όπως στο NB12 reference config.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(gradient_clip_norm))
        optimizer.step()

        pred_obs, target_obs = get_masked_vectors(
            prediction.detach(),
            batch.y.detach(),
            batch.observed_mask.detach(),
        )

        error = pred_obs - target_obs
        total_squared_error += float(torch.sum(error ** 2).item())
        total_absolute_error += float(torch.sum(torch.abs(error)).item())
        total_observed_count += int(pred_obs.numel())

    if total_observed_count <= 0:
        raise ValueError("[EMPTY TRAIN EPOCH] No observed training entries aggregated.")

    mean_mse = total_squared_error / total_observed_count
    mean_rmse = float(np.sqrt(mean_mse))
    mean_mae = total_absolute_error / total_observed_count

    return {
        "loss": mean_mse,
        "mae": mean_mae,
        "rmse": mean_rmse,
        "observed_count": total_observed_count,
    }


@torch.no_grad()
def evaluate_loader(
    model: nn.Module,
    loader,
    device: torch.device,
    collect_predictions: bool = False,
) -> dict:
    """
    Evaluation μόνο πάνω στο observed space.
    """
    model.eval()

    total_squared_error = 0.0
    total_absolute_error = 0.0
    total_observed_count = 0

    all_pred_obs = []
    all_target_obs = []

    for batch in loader:
        batch = batch.to(device)

        prediction = model(batch)
        pred_obs, target_obs = get_masked_vectors(
            prediction,
            batch.y,
            batch.observed_mask,
        )

        error = pred_obs - target_obs
        total_squared_error += float(torch.sum(error ** 2).item())
        total_absolute_error += float(torch.sum(torch.abs(error)).item())
        total_observed_count += int(pred_obs.numel())

        all_pred_obs.append(pred_obs.detach().cpu())
        all_target_obs.append(target_obs.detach().cpu())

    if total_observed_count <= 0:
        raise ValueError("[EMPTY EVAL SPACE] No observed entries aggregated in loader.")

    y_pred = torch.cat(all_pred_obs).numpy()
    y_true = torch.cat(all_target_obs).numpy()

    mean_mse = total_squared_error / total_observed_count
    mean_rmse = float(np.sqrt(mean_mse))
    mean_mae = total_absolute_error / total_observed_count
    mean_r2 = compute_r2_from_numpy(y_true=y_true, y_pred=y_pred)

    result = {
        "loss": mean_mse,
        "mae": mean_mae,
        "rmse": mean_rmse,
        "r2": mean_r2,
        "observed_count": total_observed_count,
    }

    if collect_predictions:
        result["y_true_observed"] = y_true
        result["y_pred_observed"] = y_pred

    return result


# ============================================================
# 5. Experiment helpers
# ============================================================
experiment_registry_index = experiment_registry_df.set_index("experiment_id", drop=False)


def get_experiment_row(experiment_id: str) -> pd.Series:
    if experiment_id not in experiment_registry_index.index:
        raise KeyError(f"[UNKNOWN EXPERIMENT ID] {experiment_id}")
    return experiment_registry_index.loc[experiment_id]


def build_model_and_optimizer_for_experiment(
    experiment_id: str,
    device: torch.device,
) -> tuple[nn.Module, torch.optim.Optimizer, pd.Series]:
    """
    Δημιουργεί model / optimizer ακριβώς από το frozen registry row.
    """
    row = get_experiment_row(experiment_id)

    model = GCNForecastRegressor(
        input_dim=int(row["input_feature_dim"]),
        hidden_channels=int(row["hidden_channels"]),
        dropout=float(row["dropout"]),
        message_passing_layers=int(row["message_passing_layers"]),
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(row["learning_rate"]),
        weight_decay=float(row["weight_decay"]),
    )

    return model, optimizer, row


def initialize_history_table() -> list[dict]:
    """
    Κρατά epoch-level history για export και auditability.
    """
    return []


def append_history_row(
    history_rows: list[dict],
    experiment_id: str,
    epoch: int,
    train_metrics: dict,
    val_metrics: dict,
) -> None:
    history_rows.append(
        {
            "experiment_id": experiment_id,
            "epoch": int(epoch),
            "train_loss": float(train_metrics["loss"]),
            "train_mae": float(train_metrics["mae"]),
            "train_rmse": float(train_metrics["rmse"]),
            "train_observed_count": int(train_metrics["observed_count"]),
            "val_loss": float(val_metrics["loss"]),
            "val_mae": float(val_metrics["mae"]),
            "val_rmse": float(val_metrics["rmse"]),
            "val_r2": float(val_metrics["r2"]),
            "val_observed_count": int(val_metrics["observed_count"]),
        }
    )


def run_experiment_training_skeleton(
    experiment_id: str,
    device: torch.device,
) -> dict:
    """
    Skeleton του canonical NB13 training protocol.
    Δεν το εκτελούμε ακόμη σε αυτό το section.
    """
    set_global_reproducibility(SEED)

    model, optimizer, row = build_model_and_optimizer_for_experiment(
        experiment_id=experiment_id,
        device=device,
    )

    runtime_bundle = experiment_runtime_map[experiment_id]
    train_loader = runtime_bundle["loaders"]["train"]
    val_loader = runtime_bundle["loaders"]["val"]

    early_stopping = EarlyStoppingState()
    history_rows = initialize_history_table()

    protocol = {
        "experiment_id": experiment_id,
        "topology_variant": runtime_bundle["topology_variant"],
        "message_passing_layers": int(row["message_passing_layers"]),
        "max_epochs": int(row["max_epochs"]),
        "patience": int(row["patience"]),
        "min_delta": float(row["min_delta"]),
        "gradient_clip_norm": float(row["gradient_clip_norm"]),
        "status": "ready_for_execution",
    }

    return {
        "model": model,
        "optimizer": optimizer,
        "row": row,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "early_stopping": early_stopping,
        "history_rows": history_rows,
        "protocol": protocol,
    }


# ============================================================
# 6. Sanity-check το runner skeleton χωρίς full training
# ============================================================
sanity_experiment_id = experiment_registry_df.loc[
    experiment_registry_df["enabled"],
    "experiment_id",
].iloc[0]

sanity_bundle = run_experiment_training_skeleton(
    experiment_id=sanity_experiment_id,
    device=DEVICE,
)

print("GCN model and runner skeleton initialized successfully.")
print(f"Sanity experiment_id      : {sanity_bundle['protocol']['experiment_id']}")
print(f"Topology variant          : {sanity_bundle['protocol']['topology_variant']}")
print(f"Message passing layers    : {sanity_bundle['protocol']['message_passing_layers']}")
print(f"Max epochs                : {sanity_bundle['protocol']['max_epochs']}")
print(f"Patience                  : {sanity_bundle['protocol']['patience']}")
print(f"Gradient clip norm        : {sanity_bundle['protocol']['gradient_clip_norm']}")
print(f"Train loader batches      : {len(sanity_bundle['train_loader'])}")
print(f"Validation loader batches : {len(sanity_bundle['val_loader'])}")
print(f"Model class               : {type(sanity_bundle['model']).__name__}")

GCN model and runner skeleton initialized successfully.
Sanity experiment_id      : NB13_E01
Topology variant          : self_only_control
Message passing layers    : 1
Max epochs                : 30
Patience                  : 6
Gradient clip norm        : 1.0
Train loader batches      : 489
Validation loader batches : 45
Model class               : GCNForecastRegressor


## 7. Canonical full training loop, early stopping execution και per-experiment result collection

Σε αυτό το section εκτελείται το actual `NB13` experiment plan όπως έχει ήδη
κλειδωθεί στο registry.

### Τι κάνει το section

Για κάθε enabled experiment:

- δημιουργεί model / optimizer από το frozen registry,
- τρέχει training με early stopping,
- χρησιμοποιεί το validation split μόνο για model selection,
- επαναφορτώνει το best validation checkpoint,
- και υπολογίζει final metrics μόνο στο test split.

### Export discipline

Το section αποθηκεύει:

- epoch-level training history,
- validation summary,
- final test metrics.

Το full observed-only prediction export κρατιέται προαιρετικό,
ώστε το notebook να παραμείνει πρακτικό και να μην παράγει άσκοπα
πολύ μεγάλα local artifacts σε κάθε run.

In [8]:
from __future__ import annotations

from pathlib import Path
from copy import deepcopy

import pandas as pd
import torch


# ============================================================
# 1. Runtime export policy
# ============================================================
# Για το πρώτο canonical run του NB13 κρατάμε compact exports.
# Το full observed-only prediction export μπορεί να ενεργοποιηθεί
# αργότερα μόνο αν χρειαστεί για deeper diagnostics.
EXPORT_FULL_TEST_PREDICTIONS = False


# ============================================================
# 2. Helper για observed-only prediction export (optional)
# ============================================================
@torch.no_grad()
def collect_observed_test_predictions(
    model: torch.nn.Module,
    loader,
    device: torch.device,
    experiment_id: str,
    topology_variant: str,
    message_passing_layers: int,
    n_nodes: int,
    timestamp_lookup: list,
) -> pd.DataFrame:
    """
    Συλλέγει observed-only test predictions με minimal metadata.

    Προσοχή:
    Αυτό το export μπορεί να γίνει μεγάλο, γι' αυτό το κρατάμε optional.
    """
    model.eval()

    rows = []

    for batch in loader:
        batch = batch.to(device)

        prediction = model(batch)
        mask = ensure_boolean_mask(batch.observed_mask)

        # batch.y έχει flatten nodes από όλα τα graphs του batch
        total_nodes = batch.y.shape[0]
        node_idx_within_graph = torch.arange(total_nodes, device=device) % int(n_nodes)

        batch_vector = batch.batch
        time_index_per_graph = batch.time_index.view(-1)
        time_index_per_node = time_index_per_graph[batch_vector]

        pred_obs = prediction[mask].detach().cpu().numpy()
        y_true_obs = batch.y[mask].detach().cpu().numpy()
        baseline_obs = batch.baseline_reference[mask].detach().cpu().numpy()
        time_index_obs = time_index_per_node[mask].detach().cpu().numpy()
        node_idx_obs = node_idx_within_graph[mask].detach().cpu().numpy()

        for t_idx, node_idx, y_true, y_pred, baseline in zip(
            time_index_obs,
            node_idx_obs,
            y_true_obs,
            pred_obs,
            baseline_obs,
        ):
            rows.append(
                {
                    "experiment_id": experiment_id,
                    "topology_variant": topology_variant,
                    "message_passing_layers": int(message_passing_layers),
                    "time_index": int(t_idx),
                    "timestamp": str(timestamp_lookup[int(t_idx)]),
                    "node_idx": int(node_idx),
                    "y_true_observed": float(y_true),
                    "y_pred_observed": float(y_pred),
                    "baseline_reference_observed": float(baseline),
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# 3. Single-experiment execution
# ============================================================
def execute_single_nb13_experiment(
    experiment_id: str,
    device: torch.device,
    export_full_test_predictions: bool = False,
) -> dict:
    """
    Εκτελεί ένα experiment end-to-end:
    training -> early stopping -> best model restore -> final evaluation.
    """
    # Reproducibility reset πριν από κάθε experiment
    set_global_reproducibility(SEED)

    bundle = run_experiment_training_skeleton(
        experiment_id=experiment_id,
        device=device,
    )

    model = bundle["model"]
    optimizer = bundle["optimizer"]
    row = bundle["row"]
    train_loader = bundle["train_loader"]
    val_loader = bundle["val_loader"]

    runtime_bundle = experiment_runtime_map[experiment_id]
    test_loader = runtime_bundle["loaders"]["test"]
    test_dataset_local = runtime_bundle["datasets"]["test"]

    early_stopping = bundle["early_stopping"]
    history_rows = bundle["history_rows"]

    max_epochs = int(row["max_epochs"])
    patience = int(row["patience"])
    min_delta = float(row["min_delta"])
    gradient_clip_norm = float(row["gradient_clip_norm"])

    print("-" * 80)
    print(
        f"Running {experiment_id} | "
        f"topology={runtime_bundle['topology_variant']} | "
        f"layers={int(row['message_passing_layers'])}"
    )

    final_epoch_ran = 0

    for epoch in range(1, max_epochs + 1):
        final_epoch_ran = epoch

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            gradient_clip_norm=gradient_clip_norm,
        )

        val_metrics = evaluate_loader(
            model=model,
            loader=val_loader,
            device=device,
            collect_predictions=False,
        )

        append_history_row(
            history_rows=history_rows,
            experiment_id=experiment_id,
            epoch=epoch,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
        )

        improved = early_stopping.update(
            epoch=epoch,
            current_val_loss=val_metrics["loss"],
            model=model,
            min_delta=min_delta,
        )

        # Compact progress logging χωρίς υπερβολικό notebook noise
        if epoch == 1 or improved or epoch % 5 == 0:
            print(
                f"Epoch {epoch:02d} | "
                f"train_loss={train_metrics['loss']:.6f} | "
                f"val_loss={val_metrics['loss']:.6f} | "
                f"best_val={early_stopping.best_val_loss:.6f}"
            )

        if early_stopping.epochs_without_improvement >= patience:
            print(
                f"Early stopping triggered at epoch={epoch} "
                f"(patience={patience})."
            )
            break

    if early_stopping.best_model_state is None:
        raise ValueError(f"[NO BEST CHECKPOINT] {experiment_id}")

    # Επαναφορά best validation checkpoint πριν από final reporting
    model.load_state_dict(early_stopping.best_model_state)

    best_val_metrics = evaluate_loader(
        model=model,
        loader=val_loader,
        device=device,
        collect_predictions=False,
    )

    final_test_metrics = evaluate_loader(
        model=model,
        loader=test_loader,
        device=device,
        collect_predictions=False,
    )

    history_df = pd.DataFrame(history_rows)

    validation_summary_row = {
        "experiment_id": experiment_id,
        "topology_variant": runtime_bundle["topology_variant"],
        "message_passing_layers": int(row["message_passing_layers"]),
        "best_epoch": int(early_stopping.best_epoch),
        "best_val_loss": float(early_stopping.best_val_loss),
        "val_loss_at_best_model": float(best_val_metrics["loss"]),
        "val_mae": float(best_val_metrics["mae"]),
        "val_rmse": float(best_val_metrics["rmse"]),
        "val_r2": float(best_val_metrics["r2"]),
        "val_observed_count": int(best_val_metrics["observed_count"]),
        "epochs_ran": int(final_epoch_ran),
        "stopped_early": bool(final_epoch_ran < max_epochs),
        "status": "completed",
    }

    test_metrics_row = {
        "experiment_id": experiment_id,
        "topology_variant": runtime_bundle["topology_variant"],
        "message_passing_layers": int(row["message_passing_layers"]),
        "best_epoch": int(early_stopping.best_epoch),
        "reference_nb12_best_val_loss": float(row["reference_nb12_best_val_loss"]),
        "test_loss": float(final_test_metrics["loss"]),
        "test_mae": float(final_test_metrics["mae"]),
        "test_rmse": float(final_test_metrics["rmse"]),
        "test_r2": float(final_test_metrics["r2"]),
        "test_observed_count": int(final_test_metrics["observed_count"]),
        "status": "completed",
    }

    test_predictions_df = None
    if export_full_test_predictions:
        test_predictions_df = collect_observed_test_predictions(
            model=model,
            loader=test_loader,
            device=device,
            experiment_id=experiment_id,
            topology_variant=runtime_bundle["topology_variant"],
            message_passing_layers=int(row["message_passing_layers"]),
            n_nodes=expected_n_nodes,
            timestamp_lookup=test_dataset_local.package["timestamps"],
        )

    return {
        "experiment_id": experiment_id,
        "validation_summary_row": validation_summary_row,
        "test_metrics_row": test_metrics_row,
        "history_df": history_df,
        "test_predictions_df": test_predictions_df,
        "best_model_state": deepcopy(early_stopping.best_model_state),
    }


# ============================================================
# 4. Execute all enabled experiments
# ============================================================
enabled_experiment_ids = (
    experiment_registry_df.loc[experiment_registry_df["enabled"], "experiment_id"]
    .astype(str)
    .tolist()
)

if len(enabled_experiment_ids) == 0:
    raise ValueError("[NO ENABLED EXPERIMENTS] Nothing to execute.")

all_validation_rows = []
all_test_rows = []
all_history_dfs = []
all_test_prediction_dfs = []
nb13_best_model_states = {}

for experiment_id in enabled_experiment_ids:
    result = execute_single_nb13_experiment(
        experiment_id=experiment_id,
        device=DEVICE,
        export_full_test_predictions=EXPORT_FULL_TEST_PREDICTIONS,
    )

    all_validation_rows.append(result["validation_summary_row"])
    all_test_rows.append(result["test_metrics_row"])
    all_history_dfs.append(result["history_df"])
    nb13_best_model_states[experiment_id] = result["best_model_state"]

    if result["test_predictions_df"] is not None:
        all_test_prediction_dfs.append(result["test_predictions_df"])

    # Partial saves μετά από κάθε experiment για safety
    partial_validation_df = pd.DataFrame(all_validation_rows)
    partial_test_df = pd.DataFrame(all_test_rows)
    partial_history_df = pd.concat(all_history_dfs, ignore_index=True)

    partial_validation_df.to_csv(cfg.NB13_VALIDATION_SUMMARY, index=False)
    partial_test_df.to_csv(cfg.NB13_TEST_METRICS, index=False)
    partial_history_df.to_csv(cfg.NB13_TRAINING_HISTORY, index=False)

    if EXPORT_FULL_TEST_PREDICTIONS and len(all_test_prediction_dfs) > 0:
        partial_predictions_df = pd.concat(all_test_prediction_dfs, ignore_index=True)
        partial_predictions_df.to_csv(cfg.NB13_TEST_PREDICTIONS_OBSERVED_ONLY, index=False)


# ============================================================
# 5. Final result tables
# ============================================================
nb13_validation_summary_df = pd.DataFrame(all_validation_rows).sort_values(
    ["best_val_loss", "val_mae", "experiment_id"],
    ascending=[True, True, True],
).reset_index(drop=True)

nb13_test_metrics_df = pd.DataFrame(all_test_rows).sort_values(
    ["test_mae", "test_rmse", "experiment_id"],
    ascending=[True, True, True],
).reset_index(drop=True)

nb13_training_history_df = pd.concat(all_history_dfs, ignore_index=True)

nb13_validation_summary_df.to_csv(cfg.NB13_VALIDATION_SUMMARY, index=False)
nb13_test_metrics_df.to_csv(cfg.NB13_TEST_METRICS, index=False)
nb13_training_history_df.to_csv(cfg.NB13_TRAINING_HISTORY, index=False)

if EXPORT_FULL_TEST_PREDICTIONS and len(all_test_prediction_dfs) > 0:
    nb13_test_predictions_observed_only_df = pd.concat(
        all_test_prediction_dfs,
        ignore_index=True,
    )
    nb13_test_predictions_observed_only_df.to_csv(
        cfg.NB13_TEST_PREDICTIONS_OBSERVED_ONLY,
        index=False,
    )


# ============================================================
# 6. Notebook-level display
# ============================================================
display(
    nb13_validation_summary_df[
        [
            "experiment_id",
            "topology_variant",
            "message_passing_layers",
            "best_epoch",
            "best_val_loss",
            "val_mae",
            "val_rmse",
            "val_r2",
            "epochs_ran",
            "stopped_early",
        ]
    ]
)

display(
    nb13_test_metrics_df[
        [
            "experiment_id",
            "topology_variant",
            "message_passing_layers",
            "best_epoch",
            "test_mae",
            "test_rmse",
            "test_r2",
        ]
    ]
)

print("NB13 full experiment execution completed.")
print(f"Saved validation summary : {cfg.NB13_VALIDATION_SUMMARY}")
print(f"Saved test metrics       : {cfg.NB13_TEST_METRICS}")
print(f"Saved training history   : {cfg.NB13_TRAINING_HISTORY}")
if EXPORT_FULL_TEST_PREDICTIONS:
    print(f"Saved test predictions   : {cfg.NB13_TEST_PREDICTIONS_OBSERVED_ONLY}")
else:
    print("Full observed-only test prediction export was skipped by policy.")

--------------------------------------------------------------------------------
Running NB13_E01 | topology=self_only_control | layers=1
Epoch 01 | train_loss=222935.948971 | val_loss=780.777724 | best_val=780.777724
Epoch 02 | train_loss=314.719853 | val_loss=16.104343 | best_val=16.104343
Epoch 04 | train_loss=48.857959 | val_loss=4.180238 | best_val=4.180238
Epoch 05 | train_loss=12.929138 | val_loss=3.422893 | best_val=3.422893
Epoch 06 | train_loss=2.790636 | val_loss=0.407959 | best_val=0.407959
Epoch 07 | train_loss=0.261690 | val_loss=0.060512 | best_val=0.060512
Epoch 10 | train_loss=0.072820 | val_loss=0.060877 | best_val=0.060512
Early stopping triggered at epoch=13 (patience=6).
--------------------------------------------------------------------------------
Running NB13_E02 | topology=self_only_control | layers=2
Epoch 01 | train_loss=310101.310637 | val_loss=0.060923 | best_val=0.060923
Epoch 03 | train_loss=0.078900 | val_loss=0.060519 | best_val=0.060519
Epoch 05 | tra

,experiment_id,topology_variant,message_passing_layers,best_epoch,best_val_loss,val_mae,val_rmse,val_r2,epochs_ran,stopped_early
0,NB13_E06,local_pruned_graph,2,3,0.060495,0.179085,0.245957,-0.005520,9,True
1,NB13_E01,self_only_control,1,7,0.060512,0.179332,0.245993,-0.005813,13,True
2,NB13_E02,self_only_control,2,3,0.060519,0.179418,0.246005,-0.005917,9,True
3,NB13_E05,local_pruned_graph,1,7,0.060524,0.179485,0.246015,-0.005998,13,True
4,NB13_E03,canonical_nb11_graph,1,9,0.060546,0.179787,0.246061,-0.006369,15,True
5,NB13_E04,canonical_nb11_graph,2,3,0.060560,0.179971,0.246089,-0.006602,9,True


,experiment_id,topology_variant,message_passing_layers,best_epoch,test_mae,test_rmse,test_r2
0,NB13_E06,local_pruned_graph,2,3,0.218184,0.308976,-0.035027
1,NB13_E01,self_only_control,1,7,0.218322,0.308889,-0.034443
2,NB13_E02,self_only_control,2,3,0.218370,0.308859,-0.034239
3,NB13_E05,local_pruned_graph,1,7,0.218456,0.309021,-0.035327
4,NB13_E03,canonical_nb11_graph,1,9,0.218577,0.308731,-0.033381
5,NB13_E04,canonical_nb11_graph,2,3,0.218680,0.308667,-0.032956


NB13 full experiment execution completed.
Saved validation summary : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_validation_summary.csv
Saved test metrics       : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_test_metrics.csv
Saved training history   : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_training_history.csv
Full observed-only test prediction export was skipped by policy.


## 8. NB13 ablation comparison table, control deltas, depth effects και thesis-safe interpretation scaffold

Σε αυτό το section συνοψίζονται τα αποτελέσματα του `NB13`
με τρόπο κατάλληλο για benchmark-safe ερμηνεία.

### Τι θέλουμε να απαντήσουμε

1. **Βοηθά η spatial graph πληροφορία ουσιαστικά;**
2. **Είναι το αποτέλεσμα ευαίσθητο στο topology choice;**
3. **Βοηθά ή βλάπτει το deeper aggregation;**
4. **Πώς συγκρίνεται το `NB13` best run με το `NB12` reference run;**

### Μεθοδολογική στάση

Η ανάλυση του section αυτού:

- βασίζεται μόνο στα ήδη παραγμένα validation / test exports,
- δεν επηρεάζει το training,
- δεν αλλάζει benchmark artifacts,
- και αποφεύγει overclaiming.

Το ζητούμενο δεν είναι να «αποδείξουμε» graph superiority,
αλλά να τεκμηριώσουμε αν η spatial πληροφορία προσθέτει
ουσιαστική forecasting αξία στο current repository setup.

In [9]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Helper για robust metric-column inference
#    Δεν υποθέτουμε rigid schema στο NB12 reference CSV.
# ============================================================
def infer_metric_column(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(
        f"[MISSING METRIC COLUMN] Could not infer {label}. "
        f"Available columns={df.columns.tolist()}"
    )


def build_nb12_reference_row(
    nb12_metrics_df: pd.DataFrame,
    nb12_run_config: dict,
) -> pd.DataFrame:
    """
    Φτιάχνει ένα καθαρό single-row NB12 reference table
    με canonical metric names για downstream comparison.
    """
    if nb12_metrics_df.empty:
        raise ValueError("[EMPTY NB12 TEST METRICS]")

    mae_col = infer_metric_column(
        nb12_metrics_df,
        ["test_mae", "mae", "MAE"],
        "NB12 test MAE",
    )
    rmse_col = infer_metric_column(
        nb12_metrics_df,
        ["test_rmse", "rmse", "RMSE"],
        "NB12 test RMSE",
    )
    r2_col = infer_metric_column(
        nb12_metrics_df,
        ["test_r2", "r2", "R2", "R²"],
        "NB12 test R2",
    )

    # Αν υπάρχει model-name column, προτιμάμε row που μοιάζει με GCN / NB12.
    preferred_row = nb12_metrics_df.iloc[0].copy()

    model_name_candidates = ["model", "model_name", "Model", "Model_Name"]
    model_name_col = None
    for col in model_name_candidates:
        if col in nb12_metrics_df.columns:
            model_name_col = col
            break

    if model_name_col is not None:
        mask = (
            nb12_metrics_df[model_name_col]
            .astype(str)
            .str.lower()
            .str.contains("gcn|nb12", regex=True)
        )
        if mask.any():
            preferred_row = nb12_metrics_df.loc[mask].iloc[0].copy()

    return pd.DataFrame(
        [
            {
                "reference_label": "NB12_REFERENCE",
                "reference_best_epoch": int(nb12_run_config.get("best_epoch", -1)),
                "reference_best_val_loss": float(nb12_run_config.get("best_val_loss", np.nan)),
                "reference_test_mae": float(preferred_row[mae_col]),
                "reference_test_rmse": float(preferred_row[rmse_col]),
                "reference_test_r2": float(preferred_row[r2_col]),
            }
        ]
    )


# ============================================================
# 2. Canonical merged NB13 comparison table
# ============================================================
nb13_comparison_df = (
    nb13_test_metrics_df.merge(
        nb13_validation_summary_df[
            [
                "experiment_id",
                "best_val_loss",
                "val_mae",
                "val_rmse",
                "val_r2",
                "epochs_ran",
                "stopped_early",
            ]
        ],
        on="experiment_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["test_mae", "test_rmse", "experiment_id"], ascending=[True, True, True])
    .reset_index(drop=True)
)

# Ranking fields για thesis-safe summary table
nb13_comparison_df["test_mae_rank"] = (
    nb13_comparison_df["test_mae"].rank(method="dense", ascending=True).astype(int)
)
nb13_comparison_df["test_rmse_rank"] = (
    nb13_comparison_df["test_rmse"].rank(method="dense", ascending=True).astype(int)
)

best_nb13_test_mae = float(nb13_comparison_df["test_mae"].min())
best_nb13_test_rmse = float(nb13_comparison_df["test_rmse"].min())

nb13_comparison_df["delta_test_mae_vs_best_nb13"] = (
    nb13_comparison_df["test_mae"] - best_nb13_test_mae
)
nb13_comparison_df["delta_test_rmse_vs_best_nb13"] = (
    nb13_comparison_df["test_rmse"] - best_nb13_test_rmse
)


# ============================================================
# 3. Control comparison:
#    κάθε variant συγκρίνεται με το self_only_control στο ίδιο depth.
# ============================================================
control_reference_df = (
    nb13_comparison_df.loc[
        nb13_comparison_df["topology_variant"] == "self_only_control",
        ["message_passing_layers", "test_mae", "test_rmse", "test_r2"],
    ]
    .rename(
        columns={
            "test_mae": "self_only_test_mae",
            "test_rmse": "self_only_test_rmse",
            "test_r2": "self_only_test_r2",
        }
    )
    .reset_index(drop=True)
)

control_comparison_df = (
    nb13_comparison_df.merge(
        control_reference_df,
        on="message_passing_layers",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

control_comparison_df["delta_test_mae_vs_self_only_same_depth"] = (
    control_comparison_df["test_mae"] - control_comparison_df["self_only_test_mae"]
)
control_comparison_df["delta_test_rmse_vs_self_only_same_depth"] = (
    control_comparison_df["test_rmse"] - control_comparison_df["self_only_test_rmse"]
)
control_comparison_df["delta_test_r2_vs_self_only_same_depth"] = (
    control_comparison_df["test_r2"] - control_comparison_df["self_only_test_r2"]
)


# ============================================================
# 4. Depth-effect comparison:
#    για κάθε topology συγκρίνουμε layer-2 vs layer-1.
# ============================================================
layer1_df = (
    nb13_comparison_df.loc[
        nb13_comparison_df["message_passing_layers"] == 1,
        ["topology_variant", "test_mae", "test_rmse", "test_r2", "best_val_loss"],
    ]
    .rename(
        columns={
            "test_mae": "layer1_test_mae",
            "test_rmse": "layer1_test_rmse",
            "test_r2": "layer1_test_r2",
            "best_val_loss": "layer1_best_val_loss",
        }
    )
)

layer2_df = (
    nb13_comparison_df.loc[
        nb13_comparison_df["message_passing_layers"] == 2,
        ["topology_variant", "test_mae", "test_rmse", "test_r2", "best_val_loss"],
    ]
    .rename(
        columns={
            "test_mae": "layer2_test_mae",
            "test_rmse": "layer2_test_rmse",
            "test_r2": "layer2_test_r2",
            "best_val_loss": "layer2_best_val_loss",
        }
    )
)

depth_effect_df = (
    layer1_df.merge(
        layer2_df,
        on="topology_variant",
        how="inner",
        validate="one_to_one",
    )
    .copy()
)

# Θετικό delta σημαίνει ότι το layer-2 είναι χειρότερο στο MAE/RMSE.
# Αρνητικό delta σημαίνει improvement του deeper aggregation.
depth_effect_df["delta_test_mae_layer2_minus_layer1"] = (
    depth_effect_df["layer2_test_mae"] - depth_effect_df["layer1_test_mae"]
)
depth_effect_df["delta_test_rmse_layer2_minus_layer1"] = (
    depth_effect_df["layer2_test_rmse"] - depth_effect_df["layer1_test_rmse"]
)
depth_effect_df["delta_test_r2_layer2_minus_layer1"] = (
    depth_effect_df["layer2_test_r2"] - depth_effect_df["layer1_test_r2"]
)
depth_effect_df["delta_best_val_loss_layer2_minus_layer1"] = (
    depth_effect_df["layer2_best_val_loss"] - depth_effect_df["layer1_best_val_loss"]
)


# ============================================================
# 5. NB12 reference comparison
# ============================================================
nb12_reference_df = build_nb12_reference_row(
    nb12_metrics_df=nb12_test_metrics_df,
    nb12_run_config=nb12_run_config,
)

nb13_vs_nb12_df = (
    nb13_comparison_df.assign(reference_label="NB12_REFERENCE")
    .merge(
        nb12_reference_df,
        on="reference_label",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

nb13_vs_nb12_df["delta_test_mae_vs_nb12_reference"] = (
    nb13_vs_nb12_df["test_mae"] - nb13_vs_nb12_df["reference_test_mae"]
)
nb13_vs_nb12_df["delta_test_rmse_vs_nb12_reference"] = (
    nb13_vs_nb12_df["test_rmse"] - nb13_vs_nb12_df["reference_test_rmse"]
)
nb13_vs_nb12_df["delta_test_r2_vs_nb12_reference"] = (
    nb13_vs_nb12_df["test_r2"] - nb13_vs_nb12_df["reference_test_r2"]
)


# ============================================================
# 6. Export all comparison tables
# ============================================================
comparison_dir = Path(cfg.NB13_GRAPH_ABLATION_DIR)

nb13_comparison_df.to_csv(cfg.NB13_ABLATION_COMPARISON, index=False)

control_comparison_path = comparison_dir / "nb13_control_comparison_summary.csv"
depth_effect_path = comparison_dir / "nb13_depth_effect_summary.csv"
nb13_vs_nb12_path = comparison_dir / "nb13_vs_nb12_reference_summary.csv"

control_comparison_df.to_csv(control_comparison_path, index=False)
depth_effect_df.to_csv(depth_effect_path, index=False)
nb13_vs_nb12_df.to_csv(nb13_vs_nb12_path, index=False)


# ============================================================
# 7. Compact displays for notebook interpretation
# ============================================================
display(
    nb13_comparison_df[
        [
            "experiment_id",
            "topology_variant",
            "message_passing_layers",
            "best_val_loss",
            "test_mae",
            "test_rmse",
            "test_r2",
            "test_mae_rank",
            "delta_test_mae_vs_best_nb13",
        ]
    ]
)

display(
    control_comparison_df[
        [
            "experiment_id",
            "topology_variant",
            "message_passing_layers",
            "test_mae",
            "self_only_test_mae",
            "delta_test_mae_vs_self_only_same_depth",
            "delta_test_rmse_vs_self_only_same_depth",
            "delta_test_r2_vs_self_only_same_depth",
        ]
    ].sort_values(
        ["message_passing_layers", "test_mae"],
        ascending=[True, True],
    ).reset_index(drop=True)
)

display(
    depth_effect_df[
        [
            "topology_variant",
            "layer1_test_mae",
            "layer2_test_mae",
            "delta_test_mae_layer2_minus_layer1",
            "layer1_test_rmse",
            "layer2_test_rmse",
            "delta_test_rmse_layer2_minus_layer1",
            "delta_test_r2_layer2_minus_layer1",
        ]
    ]
)

display(
    nb13_vs_nb12_df[
        [
            "experiment_id",
            "topology_variant",
            "message_passing_layers",
            "test_mae",
            "reference_test_mae",
            "delta_test_mae_vs_nb12_reference",
            "delta_test_rmse_vs_nb12_reference",
            "delta_test_r2_vs_nb12_reference",
        ]
    ].sort_values(
        ["delta_test_mae_vs_nb12_reference", "experiment_id"],
        ascending=[True, True],
    ).reset_index(drop=True)
)


# ============================================================
# 8. Notebook-level preliminary interpretation scaffold
# ============================================================
best_row = nb13_comparison_df.iloc[0].copy()

print("NB13 ablation comparison tables created successfully.")
print("-" * 72)
print(f"Best NB13 run                : {best_row['experiment_id']}")
print(f"Best topology                : {best_row['topology_variant']}")
print(f"Best message-passing layers  : {int(best_row['message_passing_layers'])}")
print(f"Best test MAE                : {float(best_row['test_mae']):.6f}")
print(f"Best test RMSE               : {float(best_row['test_rmse']):.6f}")
print(f"Best test R2                 : {float(best_row['test_r2']):.6f}")
print(f"Saved main comparison table  : {cfg.NB13_ABLATION_COMPARISON}")
print(f"Saved control comparison     : {control_comparison_path}")
print(f"Saved depth-effect summary   : {depth_effect_path}")
print(f"Saved NB12 reference summary : {nb13_vs_nb12_path}")

,experiment_id,topology_variant,message_passing_layers,best_val_loss,test_mae,test_rmse,test_r2,test_mae_rank,delta_test_mae_vs_best_nb13
0,NB13_E06,local_pruned_graph,2,0.060495,0.218184,0.308976,-0.035027,1,0.000000
1,NB13_E01,self_only_control,1,0.060512,0.218322,0.308889,-0.034443,2,0.000138
2,NB13_E02,self_only_control,2,0.060519,0.218370,0.308859,-0.034239,3,0.000187
3,NB13_E05,local_pruned_graph,1,0.060524,0.218456,0.309021,-0.035327,4,0.000272
4,NB13_E03,canonical_nb11_graph,1,0.060546,0.218577,0.308731,-0.033381,5,0.000393
5,NB13_E04,canonical_nb11_graph,2,0.060560,0.218680,0.308667,-0.032956,6,0.000497


,experiment_id,topology_variant,message_passing_layers,test_mae,self_only_test_mae,delta_test_mae_vs_self_only_same_depth,delta_test_rmse_vs_self_only_same_depth,delta_test_r2_vs_self_only_same_depth
0,NB13_E01,self_only_control,1,0.218322,0.218322,0.000000,0.000000,0.000000
1,NB13_E05,local_pruned_graph,1,0.218456,0.218322,0.000134,0.000132,-0.000884
2,NB13_E03,canonical_nb11_graph,1,0.218577,0.218322,0.000255,-0.000159,0.001062
3,NB13_E06,local_pruned_graph,2,0.218184,0.218370,-0.000187,0.000118,-0.000788
4,NB13_E02,self_only_control,2,0.218370,0.218370,0.000000,0.000000,0.000000
5,NB13_E04,canonical_nb11_graph,2,0.218680,0.218370,0.000310,-0.000192,0.001283


,topology_variant,layer1_test_mae,layer2_test_mae,delta_test_mae_layer2_minus_layer1,layer1_test_rmse,layer2_test_rmse,delta_test_rmse_layer2_minus_layer1,delta_test_r2_layer2_minus_layer1
0,self_only_control,0.218322,0.218370,0.000049,0.308889,0.308859,-0.000030,0.000204
1,local_pruned_graph,0.218456,0.218184,-0.000272,0.309021,0.308976,-0.000045,0.000300
2,canonical_nb11_graph,0.218577,0.218680,0.000104,0.308731,0.308667,-0.000063,0.000425


,experiment_id,topology_variant,message_passing_layers,test_mae,reference_test_mae,delta_test_mae_vs_nb12_reference,delta_test_rmse_vs_nb12_reference,delta_test_r2_vs_nb12_reference
0,NB13_E06,local_pruned_graph,2,0.218184,0.217742,0.000441,-0.000286,0.001916
1,NB13_E01,self_only_control,1,0.218322,0.217742,0.000579,-0.000373,0.002500
2,NB13_E02,self_only_control,2,0.218370,0.217742,0.000628,-0.000403,0.002704
3,NB13_E05,local_pruned_graph,1,0.218456,0.217742,0.000713,-0.000241,0.001616
4,NB13_E03,canonical_nb11_graph,1,0.218577,0.217742,0.000834,-0.000532,0.003562
5,NB13_E04,canonical_nb11_graph,2,0.218680,0.217742,0.000938,-0.000595,0.003986


NB13 ablation comparison tables created successfully.
------------------------------------------------------------------------
Best NB13 run                : NB13_E06
Best topology                : local_pruned_graph
Best message-passing layers  : 2
Best test MAE                : 0.218184
Best test RMSE               : 0.308976
Best test R2                 : -0.035027
Saved main comparison table  : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_ablation_comparison.csv
Saved control comparison     : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_control_comparison_summary.csv
Saved depth-effect summary   : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_depth_effect_summary.csv
Saved NB12 reference summary : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\

## 9. Final findings, limitations και scope boundary

### Τι έδειξε το `NB13`

Το `NB13` εκτελέστηκε ως **strict follow-up** του `NB12`
με στόχο ένα controlled **graph ablation / spatial sensitivity analysis**
και όχι νέο broad graph benchmark.

Με βάση τα current `NB13` runs, προκύπτουν τα εξής:

1. **Best `NB13` run**
   - Το καλύτερο run του `NB13` ήταν το:
     - **`NB13_E06`**
     - `topology_variant = local_pruned_graph`
     - `message_passing_layers = 2`
   - Με final test metrics:
     - **MAE = 0.218184**
     - **RMSE = 0.308976**
     - **R² = -0.035027**

2. **Spatial information vs self-only control**
   - Το `local_pruned_graph` με 2 layers πέτυχε την καλύτερη επίδοση εντός του `NB13`,
     αλλά η διαφορά του από το αντίστοιχο `self_only_control` στο ίδιο depth ήταν πολύ μικρή:
     - `delta_test_mae_vs_self_only_same_depth = -0.000187`
   - Σε 1-layer setting, το `local_pruned_graph` ήταν ελαφρώς χειρότερο από το `self_only_control`:
     - `delta_test_mae_vs_self_only_same_depth = +0.000134`
   - Άρα το current result **δεν** δείχνει ισχυρό ή σταθερό spatial gain.

3. **Topology sensitivity**
   - Το `canonical_nb11_graph` ήταν consistently το χειρότερο topology family στο test MAE.
   - Το `local_pruned_graph` απέδωσε καλύτερα από το canonical graph,
     κάτι που υποδηλώνει ότι το αποτέλεσμα είναι **ευαίσθητο στην topology choice**.
   - Παρ' όλα αυτά, το μέγεθος της βελτίωσης παραμένει μικρό.

4. **Aggregation depth effects**
   - Στο `self_only_control`, η μετάβαση από 1 σε 2 layers ήταν ελαφρώς χειρότερη στο test MAE:
     - `delta_test_mae_layer2_minus_layer1 = +0.000049`
   - Στο `local_pruned_graph`, τα 2 layers ήταν ελαφρώς καλύτερα:
     - `delta_test_mae_layer2_minus_layer1 = -0.000272`
   - Στο `canonical_nb11_graph`, τα 2 layers ήταν ελαφρώς χειρότερα:
     - `delta_test_mae_layer2_minus_layer1 = +0.000104`
   - Άρα το deeper aggregation **δεν** προσφέρει γενικά και σταθερά όφελος.
     Το effect φαίνεται να εξαρτάται από το topology.

5. **Comparison against the `NB12` reference**
   - Κανένα `NB13` run δεν βελτίωσε το `NB12` reference στο test MAE.
   - Για το best `NB13` run (`NB13_E06`), η διαφορά ήταν:
     - `delta_test_mae_vs_nb12_reference = +0.000441`
   - Άρα το `NB13` **δεν** τεκμηριώνει υπεροχή έναντι του `NB12` reference run
     με βάση το primary ranking criterion του benchmark.

### Συνολικό methodological συμπέρασμα

Το `NB13` υποστηρίζει την εξής προσεκτική ερμηνεία:

- η spatial graph πληροφορία **μπορεί** να επηρεάζει την επίδοση,
- το αποτέλεσμα είναι πράγματι **ευαίσθητο στο topology design**,
- ένα πιο local / pruned graph φαίνεται προτιμότερο από το πλήρες canonical graph,
- αλλά το μέγεθος του observed gain είναι **πολύ μικρό**,
- και δεν αρκεί για claim ουσιαστικής graph superiority στο current repository setup.

Με άλλα λόγια, το current evidence είναι πιο συμβατό με το ότι:

- το forecasting task κυριαρχείται σε μεγάλο βαθμό από ήδη ισχυρό node-local / tabular signal,
- ενώ η neighbor aggregation προσθέτει το πολύ μικρό και topology-sensitive incremental effect.

### Τι ΔΕΝ έδειξε το `NB13`

Το `NB13` **δεν** έδειξε ότι:

- τα graph models υπερέχουν καθαρά των simpler controls,
- το deeper message passing είναι γενικά καλύτερο,
- η canonical graph topology είναι η σωστή επιλογή για το current task,
- ή ότι υπάρχει ήδη ισχυρή empirical βάση για broader GNN superiority claims.

### Scope boundary

Το notebook παραμένει αυστηρά:

- forecasting-first,
- benchmark-safe,
- validation-driven για model selection,
- test-only για final reporting,
- και non-overclaiming.

Δεν αποτελεί:

- νέο broad graph benchmark,
- proof of graph superiority,
- sequence-model stage,
- PHM stage,
- ή anomaly / fault diagnosis module.

### Future work boundary

Με βάση τα current findings, το πιο defensible future direction δεν είναι άμεσο broader claim,
αλλά ένα προσεκτικό επόμενο βήμα όπως:

- πιο systematic local-topology design exploration,
- stronger edge weighting / distance-aware message passing,
- ή μελλοντική σύγκριση με richer graph formulations,

πάντα χωρίς να συγχέεται αυτό με ήδη αποδεδειγμένο graph advantage.

In [10]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Συγκεντρώνουμε ένα compact notebook summary
# ============================================================
best_nb13_row = nb13_comparison_df.iloc[0].copy()

# Self-only controls ανά depth
self_only_depth1 = control_comparison_df[
    (control_comparison_df["topology_variant"] == "self_only_control") &
    (control_comparison_df["message_passing_layers"] == 1)
].iloc[0]

self_only_depth2 = control_comparison_df[
    (control_comparison_df["topology_variant"] == "self_only_control") &
    (control_comparison_df["message_passing_layers"] == 2)
].iloc[0]

# Best local-pruned row
best_local_pruned_row = nb13_comparison_df[
    nb13_comparison_df["topology_variant"] == "local_pruned_graph"
].sort_values(["test_mae", "experiment_id"], ascending=[True, True]).iloc[0]

# Canonical graph family best row
best_canonical_row = nb13_comparison_df[
    nb13_comparison_df["topology_variant"] == "canonical_nb11_graph"
].sort_values(["test_mae", "experiment_id"], ascending=[True, True]).iloc[0]

# Depth-effect table rows
depth_self_only = depth_effect_df[
    depth_effect_df["topology_variant"] == "self_only_control"
].iloc[0]

depth_local_pruned = depth_effect_df[
    depth_effect_df["topology_variant"] == "local_pruned_graph"
].iloc[0]

depth_canonical = depth_effect_df[
    depth_effect_df["topology_variant"] == "canonical_nb11_graph"
].iloc[0]

# NB12 reference comparison για το best NB13 run
best_vs_nb12_row = nb13_vs_nb12_df[
    nb13_vs_nb12_df["experiment_id"] == best_nb13_row["experiment_id"]
].iloc[0]


# ============================================================
# 2. Notebook summary JSON
# ============================================================
nb13_summary_payload = {
    "notebook": "13_graph_ablation_and_spatial_sensitivity_analysis.ipynb",
    "status": "completed",
    "scope": "graph_ablation_and_spatial_sensitivity",
    "role": "strict_follow_up_of_nb12",
    "selection_rule": "validation_only_for_model_selection",
    "reporting_rule": "test_only_for_final_reporting",
    "benchmark_reference_mode": "read_only",
    "enabled_experiments": enabled_experiment_ids,
    "best_nb13_run": {
        "experiment_id": str(best_nb13_row["experiment_id"]),
        "topology_variant": str(best_nb13_row["topology_variant"]),
        "message_passing_layers": int(best_nb13_row["message_passing_layers"]),
        "best_val_loss": float(best_nb13_row["best_val_loss"]),
        "test_mae": float(best_nb13_row["test_mae"]),
        "test_rmse": float(best_nb13_row["test_rmse"]),
        "test_r2": float(best_nb13_row["test_r2"]),
    },
    "control_comparison": {
        "best_local_pruned_vs_self_only_same_depth_delta_test_mae": float(
            best_local_pruned_row["test_mae"] - float(self_only_depth2["test_mae"])
        ) if int(best_local_pruned_row["message_passing_layers"]) == 2 else float("nan"),
        "best_canonical_vs_self_only_same_depth_delta_test_mae": float(
            best_canonical_row["test_mae"] - (
                float(self_only_depth1["test_mae"])
                if int(best_canonical_row["message_passing_layers"]) == 1
                else float(self_only_depth2["test_mae"])
            )
        ),
    },
    "depth_effects": {
        "self_only_delta_test_mae_layer2_minus_layer1": float(
            depth_self_only["delta_test_mae_layer2_minus_layer1"]
        ),
        "local_pruned_delta_test_mae_layer2_minus_layer1": float(
            depth_local_pruned["delta_test_mae_layer2_minus_layer1"]
        ),
        "canonical_graph_delta_test_mae_layer2_minus_layer1": float(
            depth_canonical["delta_test_mae_layer2_minus_layer1"]
        ),
    },
    "nb12_reference_comparison": {
        "reference_test_mae": float(best_vs_nb12_row["reference_test_mae"]),
        "best_nb13_delta_test_mae_vs_nb12_reference": float(
            best_vs_nb12_row["delta_test_mae_vs_nb12_reference"]
        ),
        "best_nb13_delta_test_rmse_vs_nb12_reference": float(
            best_vs_nb12_row["delta_test_rmse_vs_nb12_reference"]
        ),
        "best_nb13_delta_test_r2_vs_nb12_reference": float(
            best_vs_nb12_row["delta_test_r2_vs_nb12_reference"]
        ),
    },
    "high_level_interpretation": [
        "The result is sensitive to topology choice.",
        "Local pruning performs slightly better than the canonical graph.",
        "The observed spatial gain is small and not sufficient for strong superiority claims.",
        "No NB13 run improves on the NB12 reference in test MAE.",
    ],
}


# ============================================================
# 3. Export manifest
# ============================================================
export_manifest_rows = [
    {
        "artifact_name": "nb13_experiment_registry",
        "path": str(cfg.NB13_EXPERIMENT_REGISTRY),
        "status": "created",
    },
    {
        "artifact_name": "nb13_run_config",
        "path": str(cfg.NB13_RUN_CONFIG),
        "status": "created",
    },
    {
        "artifact_name": "nb13_validation_summary",
        "path": str(cfg.NB13_VALIDATION_SUMMARY),
        "status": "created",
    },
    {
        "artifact_name": "nb13_test_metrics",
        "path": str(cfg.NB13_TEST_METRICS),
        "status": "created",
    },
    {
        "artifact_name": "nb13_training_history",
        "path": str(cfg.NB13_TRAINING_HISTORY),
        "status": "created",
    },
    {
        "artifact_name": "nb13_ablation_comparison",
        "path": str(cfg.NB13_ABLATION_COMPARISON),
        "status": "created",
    },
    {
        "artifact_name": "nb13_control_comparison_summary",
        "path": str(control_comparison_path),
        "status": "created",
    },
    {
        "artifact_name": "nb13_depth_effect_summary",
        "path": str(depth_effect_path),
        "status": "created",
    },
    {
        "artifact_name": "nb13_vs_nb12_reference_summary",
        "path": str(nb13_vs_nb12_path),
        "status": "created",
    },
]

if EXPORT_FULL_TEST_PREDICTIONS:
    export_manifest_rows.append(
        {
            "artifact_name": "nb13_test_predictions_observed_only",
            "path": str(cfg.NB13_TEST_PREDICTIONS_OBSERVED_ONLY),
            "status": "created",
        }
    )
else:
    export_manifest_rows.append(
        {
            "artifact_name": "nb13_test_predictions_observed_only",
            "path": str(cfg.NB13_TEST_PREDICTIONS_OBSERVED_ONLY),
            "status": "skipped_by_policy",
        }
    )

nb13_export_manifest_df = pd.DataFrame(export_manifest_rows)
nb13_export_manifest_df.to_csv(cfg.NB13_EXPORT_MANIFEST, index=False)

summary_json_path = Path(cfg.NB13_GRAPH_ABLATION_DIR) / "nb13_notebook_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(nb13_summary_payload, f, ensure_ascii=False, indent=2)


# ============================================================
# 4. Προβολή manifest για notebook closure
# ============================================================
display(nb13_export_manifest_df)

print("NB13 notebook closure artifacts saved successfully.")
print(f"Saved export manifest : {cfg.NB13_EXPORT_MANIFEST}")
print(f"Saved notebook summary: {summary_json_path}")

,artifact_name,path,status
0,nb13_experiment_registry,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
1,nb13_run_config,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
2,nb13_validation_summary,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
3,nb13_test_metrics,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
4,nb13_training_history,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
5,nb13_ablation_comparison,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
6,nb13_control_comparison_summary,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
7,nb13_depth_effect_summary,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
8,nb13_vs_nb12_reference_summary,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,created
9,nb13_test_predictions_observed_only,C:\Users\diony\Desktop\WindPower_DigitalTwin\d...,skipped_by_policy


NB13 notebook closure artifacts saved successfully.
Saved export manifest : C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_export_manifest.csv
Saved notebook summary: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\graph_baselines\nb13_graph_ablation_and_spatial_sensitivity\nb13_notebook_summary.json


## 10. Final sanity check και canonical conclusion

### Final sanity check

Πριν θεωρηθεί το `NB13` ολοκληρωμένο, γίνεται ένα τελικό consistency pass
πάνω στα exported artifacts και στα βασικά notebook-level findings.

Ο στόχος δεν είναι νέο evaluation ή νέο training,
αλλά επιβεβαίωση ότι:

- όλα τα κύρια exports υπάρχουν,
- τα experiment ids είναι συνεπή across outputs,
- το ranking είναι consistent με το exported comparison table,
- και το notebook closure είναι συμβατό με το actual empirical result.

### Canonical conclusion

Το `NB13` ολοκληρώθηκε ως **strict graph ablation / spatial sensitivity follow-up**
του `NB12`.

Το βασικό συμπέρασμα είναι ότι:

- η επίδοση είναι **ευαίσθητη στην topology choice**,
- ένα **local-pruned graph** αποδίδει ελαφρώς καλύτερα από το πλήρες canonical graph,
- αλλά το observed gain έναντι του `self_only_control` είναι **πολύ μικρό**,
- και κανένα `NB13` run **δεν** βελτιώνει το `NB12` reference στο primary benchmark criterion (**test MAE**).

Άρα το `NB13` **δεν** τεκμηριώνει ισχυρή ή σταθερή graph superiority
στο current repository setup.

Η πιο defensible ερμηνεία είναι ότι:

- το forecasting task κυριαρχείται σε μεγάλο βαθμό από ισχυρό node-local / tabular signal,
- ενώ η neighbor aggregation προσθέτει, στην καλύτερη περίπτωση,
  μικρό και topology-sensitive incremental effect.

Το notebook παραμένει αυστηρά:

- forecasting-first,
- benchmark-safe,
- validation-driven για selection,
- test-only για final reporting,
- και non-overclaiming.

In [11]:
from pathlib import Path

import pandas as pd


# ============================================================
# 1. Required final exports
# ============================================================
required_final_exports = {
    "NB13_EXPERIMENT_REGISTRY": cfg.NB13_EXPERIMENT_REGISTRY,
    "NB13_RUN_CONFIG": cfg.NB13_RUN_CONFIG,
    "NB13_VALIDATION_SUMMARY": cfg.NB13_VALIDATION_SUMMARY,
    "NB13_TEST_METRICS": cfg.NB13_TEST_METRICS,
    "NB13_TRAINING_HISTORY": cfg.NB13_TRAINING_HISTORY,
    "NB13_ABLATION_COMPARISON": cfg.NB13_ABLATION_COMPARISON,
    "NB13_EXPORT_MANIFEST": cfg.NB13_EXPORT_MANIFEST,
}

for label, path in required_final_exports.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"[MISSING FINAL EXPORT] {label}: {path}")


# ============================================================
# 2. Reload exported tables for end-of-notebook consistency
# ============================================================
final_registry_df = pd.read_csv(cfg.NB13_EXPERIMENT_REGISTRY)
final_validation_df = pd.read_csv(cfg.NB13_VALIDATION_SUMMARY)
final_test_df = pd.read_csv(cfg.NB13_TEST_METRICS)
final_history_df = pd.read_csv(cfg.NB13_TRAINING_HISTORY)
final_comparison_df = pd.read_csv(cfg.NB13_ABLATION_COMPARISON)
final_manifest_df = pd.read_csv(cfg.NB13_EXPORT_MANIFEST)


# ============================================================
# 3. Experiment-id consistency checks
# ============================================================
enabled_ids_expected = sorted(
    experiment_registry_df.loc[experiment_registry_df["enabled"], "experiment_id"].astype(str).tolist()
)
validation_ids_found = sorted(final_validation_df["experiment_id"].astype(str).unique().tolist())
test_ids_found = sorted(final_test_df["experiment_id"].astype(str).unique().tolist())
history_ids_found = sorted(final_history_df["experiment_id"].astype(str).unique().tolist())
comparison_ids_found = sorted(final_comparison_df["experiment_id"].astype(str).unique().tolist())

if enabled_ids_expected != validation_ids_found:
    raise ValueError(
        f"[VALIDATION ID MISMATCH] expected={enabled_ids_expected} found={validation_ids_found}"
    )

if enabled_ids_expected != test_ids_found:
    raise ValueError(
        f"[TEST ID MISMATCH] expected={enabled_ids_expected} found={test_ids_found}"
    )

if enabled_ids_expected != history_ids_found:
    raise ValueError(
        f"[HISTORY ID MISMATCH] expected={enabled_ids_expected} found={history_ids_found}"
    )

if enabled_ids_expected != comparison_ids_found:
    raise ValueError(
        f"[COMPARISON ID MISMATCH] expected={enabled_ids_expected} found={comparison_ids_found}"
    )


# ============================================================
# 4. Duplicate / null checks
# ============================================================
for df_name, df_obj in {
    "validation": final_validation_df,
    "test": final_test_df,
    "comparison": final_comparison_df,
}.items():
    if df_obj["experiment_id"].duplicated().any():
        dupes = df_obj.loc[df_obj["experiment_id"].duplicated(), "experiment_id"].tolist()
        raise ValueError(f"[DUPLICATE EXPERIMENT IDS] {df_name}: {dupes}")

if final_validation_df[["best_val_loss", "val_mae", "val_rmse"]].isnull().any().any():
    raise ValueError("[NULL VALIDATION METRICS]")

if final_test_df[["test_mae", "test_rmse", "test_r2"]].isnull().any().any():
    raise ValueError("[NULL TEST METRICS]")


# ============================================================
# 5. Ranking consistency checks
# ============================================================
comparison_sorted_by_mae = final_comparison_df.sort_values(
    ["test_mae", "test_rmse", "experiment_id"],
    ascending=[True, True, True],
).reset_index(drop=True)

if not comparison_sorted_by_mae["experiment_id"].equals(final_comparison_df["experiment_id"]):
    raise ValueError("[COMPARISON SORT ORDER MISMATCH] exported comparison table is not canonically sorted.")

best_comparison_row = final_comparison_df.iloc[0]
best_test_row = final_test_df.sort_values(
    ["test_mae", "test_rmse", "experiment_id"],
    ascending=[True, True, True],
).reset_index(drop=True).iloc[0]

if str(best_comparison_row["experiment_id"]) != str(best_test_row["experiment_id"]):
    raise ValueError(
        "[BEST RUN MISMATCH] comparison table best row differs from test metrics best row."
    )


# ============================================================
# 6. Expected empirical result check for the current run
# ============================================================
expected_best_experiment_id = "NB13_E06"
if str(best_comparison_row["experiment_id"]) != expected_best_experiment_id:
    raise ValueError(
        f"[UNEXPECTED BEST RUN] expected={expected_best_experiment_id}, "
        f"found={best_comparison_row['experiment_id']}"
    )

if str(best_comparison_row["topology_variant"]) != "local_pruned_graph":
    raise ValueError(
        f"[UNEXPECTED BEST TOPOLOGY] found={best_comparison_row['topology_variant']}"
    )

if int(best_comparison_row["message_passing_layers"]) != 2:
    raise ValueError(
        f"[UNEXPECTED BEST DEPTH] found={best_comparison_row['message_passing_layers']}"
    )


# ============================================================
# 7. Manifest consistency
# ============================================================
manifest_required_names = {
    "nb13_experiment_registry",
    "nb13_run_config",
    "nb13_validation_summary",
    "nb13_test_metrics",
    "nb13_training_history",
    "nb13_ablation_comparison",
    "nb13_control_comparison_summary",
    "nb13_depth_effect_summary",
    "nb13_vs_nb12_reference_summary",
    "nb13_test_predictions_observed_only",
}

manifest_names_found = set(final_manifest_df["artifact_name"].astype(str).tolist())
missing_manifest_names = sorted(manifest_required_names - manifest_names_found)
if missing_manifest_names:
    raise ValueError(f"[MANIFEST INCOMPLETE] missing={missing_manifest_names}")


# ============================================================
# 8. Compact final display
# ============================================================
sanity_summary_df = pd.DataFrame(
    [
        {
            "check_group": "exports",
            "status": "passed",
            "detail": f"{len(required_final_exports)} required final exports found",
        },
        {
            "check_group": "experiment_ids",
            "status": "passed",
            "detail": f"{len(enabled_ids_expected)} enabled experiments consistent across outputs",
        },
        {
            "check_group": "ranking",
            "status": "passed",
            "detail": f"best run = {best_comparison_row['experiment_id']}",
        },
        {
            "check_group": "best_topology",
            "status": "passed",
            "detail": (
                f"{best_comparison_row['topology_variant']} | "
                f"layers={int(best_comparison_row['message_passing_layers'])}"
            ),
        },
        {
            "check_group": "best_metrics",
            "status": "passed",
            "detail": (
                f"test_mae={float(best_comparison_row['test_mae']):.6f}, "
                f"test_rmse={float(best_comparison_row['test_rmse']):.6f}, "
                f"test_r2={float(best_comparison_row['test_r2']):.6f}"
            ),
        },
    ]
)

display(sanity_summary_df)

print("NB13 FINAL SANITY CHECK PASSED")
print("-" * 72)
print(f"Best experiment_id         : {best_comparison_row['experiment_id']}")
print(f"Best topology              : {best_comparison_row['topology_variant']}")
print(f"Best message-passing depth : {int(best_comparison_row['message_passing_layers'])}")
print(f"Best test MAE              : {float(best_comparison_row['test_mae']):.6f}")
print(f"Best test RMSE             : {float(best_comparison_row['test_rmse']):.6f}")
print(f"Best test R2               : {float(best_comparison_row['test_r2']):.6f}")

,check_group,status,detail
0,exports,passed,7 required final exports found
1,experiment_ids,passed,6 enabled experiments consistent across outputs
2,ranking,passed,best run = NB13_E06
3,best_topology,passed,local_pruned_graph | layers=2
4,best_metrics,passed,"test_mae=0.218184, test_rmse=0.308976, test_r2..."


NB13 FINAL SANITY CHECK PASSED
------------------------------------------------------------------------
Best experiment_id         : NB13_E06
Best topology              : local_pruned_graph
Best message-passing depth : 2
Best test MAE              : 0.218184
Best test RMSE             : 0.308976
Best test R2               : -0.035027


## 11. Final conclusion

Το `NB13` ολοκληρώθηκε ως **strict graph ablation / spatial sensitivity follow-up**
του `NB12`, με fixed training configuration, validation-driven model selection
και test-only final reporting.

Το καλύτερο run του `NB13` ήταν το:

- **`NB13_E06`**
- `topology_variant = local_pruned_graph`
- `message_passing_layers = 2`

με:

- **test MAE = 0.218184**
- **test RMSE = 0.308976**
- **test R² = -0.035027**

Η βασική ερμηνεία των αποτελεσμάτων είναι η εξής:

1. Η επίδοση είναι **ευαίσθητη στην topology choice**.
2. Το **local-pruned graph** αποδίδει ελαφρώς καλύτερα από το πλήρες canonical graph.
3. Η spatial neighbor information δεν δίνει **ισχυρό ή σταθερό** gain έναντι του `self_only_control`.
4. Το effect του deeper aggregation δεν είναι γενικά θετικό· εξαρτάται από το topology.
5. Κανένα `NB13` run δεν βελτιώνει το `NB12` reference στο **test MAE**.

Άρα το `NB13` **δεν** τεκμηριώνει ισχυρή graph superiority
στο current repository setup.

Η πιο defensible ερμηνεία είναι ότι το forecasting task κυριαρχείται
σε μεγάλο βαθμό από ισχυρό node-local / tabular signal,
ενώ η graph topology προσθέτει μόνο μικρό και topology-sensitive incremental effect.

Το notebook παραμένει αυστηρά:

- forecasting-first,
- benchmark-safe,
- non-overclaiming,
- και κατάλληλο ως thesis-ready sensitivity analysis stage,
όχι ως broad graph-benchmark claim.